# Evaluación experimental del agente RAG — Ley 21.719

## Estado y orden de ejecución

**El split de DESARROLLO está cerrado.** Se ejecutó sobre el commit `a7d27e5` con huella
de configuración `7b41161dedb1`. Los archivos `respuestas_desarrollo.jsonl` y
`ragas_desarrollo.jsonl` se perdieron al no haber quedado en la carpeta compartida, por
lo que esa corrida **no se puede extender ni recomputar**: regenerar las respuestas
produciría textos distintos (temperatura 0,1) y los puntajes RAGAS existentes dejarían
de corresponder.

Sus resultados sobreviven en dos lugares: las salidas guardadas de las celdas E5, E8, E9,
E10 y E11, y los CSV que escribe la celda **R · Registro** más abajo. **No borrar las
salidas de este notebook.**

### Para correr el split de PRUEBA

En E1: `SPLIT = "prueba"`, `CONFIG_CONGELADA = True`, `APLICAR_PARCHE_P1 = False`.
Dejar `UMBRAL_RERANK = 0.05` en C1 del notebook del equipo: es la configuración con la
que se midió desarrollo, y cambiarla rompería la comparabilidad entre ambos splits.

Orden: **E1 → rutas → E2 → V3 → E3 → E4 → E5 → E6 → E6-bis → E7 → E8-bis → E8 → E10**

RAGAS (E9) sobre prueba requiere ~618 llamadas al juez contra una cuota de 500 diarias,
o sea dos días. Si no alcanza, **E8 y E10 no necesitan juez** y bastan para contrastar
H1 y H2 sobre las métricas automáticas, que son las que sostienen la hipótesis principal.

No correr E9 (versión superada, celda con candado) ni E11 sobre prueba: la calibración
del umbral se hace solo con desarrollo.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# R · REGISTRO DE LOS RESULTADOS DE DESARROLLO
# ═══════════════════════════════════════════════════════════════
# Estas cifras NO se recalculan aquí: están TRANSCRITAS de la corrida del
# split de desarrollo sobre el commit a7d27e5, huella 7b41161dedb1, tal
# como quedaron impresas en las salidas de las celdas E5, E8, E9, E10 y E11.
#
# Por qué existe esta celda: los archivos respuestas_desarrollo.jsonl y
# ragas_desarrollo.jsonl se perdieron, de modo que esa corrida no se puede
# recomputar. Hasta ahora el único registro eran las salidas del notebook,
# que desaparecen si alguien las borra. Esta celda las deja como CSV en la
# carpeta compartida.
#
# Correrla no consume cuota ni GPU y no toca el split de prueba.
import pandas as pd

DEST = EVAL.DIR_RES / "registro_desarrollo"
DEST.mkdir(parents=True, exist_ok=True)
PROC = dict(split="desarrollo", commit="a7d27e5", huella="7b41161dedb1",
            generador="qwen3:8b", umbral_rerank=0.05, n_resp=45, n_fuera=5,
            fuente="transcrito de las salidas del notebook")

T = {}

T["recall_por_etapa"] = pd.DataFrame([
    ("C2 · densa@25",            75.6, "61,3–85,8", 0.467, 25.0),
    ("C2 · BM25@25",             71.1, "56,6–82,3", 0.317, 25.0),
    ("C2 · RRF@10",              62.2, "47,6–74,9", 0.439, 10.0),
    ("C2 · final (top-k)",       62.2, "47,6–74,9", 0.448,  5.3),
    ("C2 sin expansión · final", 33.3, "21,4–47,9", 0.288,  2.5),
    ("C1 · densa fija (top-k)",  60.0, "45,5–73,0", 0.396,  8.0),
], columns=["etapa", "recall_pct", "ic95", "mrr", "k_medio"])

T["recall_por_tipo"] = pd.DataFrame([
    ("coloquial",  39,  53.8,  56.4),
    ("juridica",    5, 100.0, 100.0),
    ("multinorma",  1, 100.0, 100.0),   # n=1: no admite inferencia
], columns=["tipo", "n", "C1_densa_fija_pct", "C2_final_pct"])

T["metricas_automaticas"] = pd.DataFrame([
    ("C0", "qwen3:8b",   24.0, 50, 17.8, 45, 20.0,  0.0, 10.980),
    ("C1", "qwen3:8b",    0.0, 50, 48.9, 45,  0.0,  0.0, 14.290),
    ("C2", "gemma4:e2b",  0.0, 47, 61.9, 42, 40.0,  2.4, 12.790),
    ("C2", "qwen3:8b",    0.0, 50, 62.2, 45, 40.0,  0.0, 14.745),
], columns=["condicion", "modelo", "cita_inventada_pct", "n_cita_inv",
            "cita_esperada_pct", "n_cita_esp", "abstencion_correcta_pct",
            "falsa_abstencion_pct", "latencia_mediana_s"])

T["calibracion_umbral"] = pd.DataFrame([
    (0.000, 68.9, 0.476, 0, 0), (0.005, 68.9, 0.476, 0, 3),
    (0.010, 66.7, 0.471, 0, 3), (0.020, 64.4, 0.460, 0, 3),
    (0.030, 62.2, 0.449, 2, 3), (0.050, 62.2, 0.448, 4, 3),
    (0.100, 55.6, 0.400, 7, 4),
], columns=["umbral", "recall_pct", "mrr", "respondibles_sin_fuentes",
            "fuera_corpus_sin_fuentes"])

T["hipotesis"] = pd.DataFrame([
    ("H1", "cita_inventada",   "C2 vs C0", 50, "-24,0 pp",      0.0005, 0.0034, True),
    ("H1", "cita_esperada",    "C2 vs C0", 45, "+44,4 pp",      0.0003, 0.0026, True),
    ("H1", "answer_relevancy", "C2 vs C0", 12, "+5,6 mediana",  0.0161, 0.0967, False),
    ("H2", "cita_inventada",   "C2 vs C1", 50, "+0,0 pp",       1.0000, 1.0000, False),
    ("H2", "cita_esperada",    "C2 vs C1", 45, "+13,3 pp",      0.2101, 0.8405, False),
    ("H2", "answer_relevancy", "C2 vs C1", 32, "-1,9 mediana",  0.0378, 0.1889, False),
    ("H2", "faithfulness",     "C2 vs C1", 28, "-1,5 mediana",  0.8401, 1.0000, False),
    ("H2", "recall@k",         "C2 vs C1", 45, "+2,2 pp",       1.0000, 1.0000, False),
], columns=["hipotesis", "metrica", "comparacion", "n", "diferencia",
            "p", "p_holm", "significativa"])

# ⚠️ incompleta: faltan ~25 evaluaciones de C2/qwen3 por cuota del juez,
#    y las que faltan no son aleatorias (son las últimas de la cola).
T["ragas"] = pd.DataFrame([
    ("C0", "qwen3:8b",   84.1, "79,7–88,6", 12, None, None,        0),
    ("C1", "qwen3:8b",   86.9, "84,7–89,0", 45, 57.5, "50,6–64,2", 45),
    ("C2", "gemma4:e2b", 88.8, "86,0–92,8",  4, 83.7, "75,0–93,8",  4),
    ("C2", "qwen3:8b",   75.9, "64,4–85,0", 32, 56.7, "48,8–64,4", 28),
], columns=["condicion", "modelo", "answer_relevancy", "ar_ic95", "ar_n",
            "faithfulness", "f_ic95", "f_n"])

for nombre, df in T.items():
    for k, v in PROC.items():
        df[k] = v
    ruta = DEST / f"{nombre}_desarrollo.csv"
    df.to_csv(ruta, index=False)
    print(f"💾 {ruta.name:36} {len(df)} filas")

print(f"\n✅ {len(T)} tablas en {DEST}")
print("   Transcritas, no recalculadas. Procedencia en las columnas commit/huella/fuente.")


In [2]:
# ═══════════════════════════════════════════════════════════════
# E1 · CONFIGURACIÓN DE LA EVALUACIÓN  ← único lugar que se edita
# ═══════════════════════════════════════════════════════════════
# ragas 0.4.3 importa un módulo que langchain-community 0.4 ya no trae
# (ChatVertexAI): sin este pin, "import ragas" falla. Verificado el 7-nov-2026.
%pip -q install "ragas==0.4.3" "langchain-community<0.4" scipy statsmodels

from pathlib import Path
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

class EVAL:
    # --- código del equipo ---
    REPO_URL  = "https://github.com/belensilva1206/Tesis_belen_ian_javi.git"
    NOTEBOOK  = "RAG_Leyes_PYME.ipynb"
    # None = usa la ruta escrita en C2. Si en tu Drive la carpeta "Leyes" está en
    # otra ubicación (p. ej. un acceso directo), escribe aquí la ruta completa.
    # ← EDITAR: copia congelada de los 6 PDF. Si no existe, créala antes
    #   de la corrida de prueba (ver celda V3-pre).
    RUTA_LEYES = "/content/drive/MyDrive/resultados_evaluacion_tesis/Leyes"

    # --- corrección del parser (ver sección E2) ---
    # P1 ya está fusionado en C3 del repositorio del equipo, así que este
    # parche es redundante. Déjalo en True hasta comprobar la equivalencia
    # con V3; después ponlo en False para que haya una sola fuente de verdad.
    APLICAR_PARCHE_P1 = True

    # --- banco de preguntas ---
    # ← EDITAR: el banco y los resultados viven en la carpeta COMPARTIDA
    #   del equipo, no en el Drive personal de cada uno.
    BANCO  = Path("/content/drive/MyDrive/resultados_evaluacion_tesis/banco_etiquetado.csv")
    SPLIT             = "prueba"
    CONFIG_CONGELADA  = True
    APLICAR_PARCHE_P1 = False                  # >0: solo las primeras N preguntas (medir tiempos antes de la corrida)

    # --- condiciones ---
    GENERADOR_FAMILIA = "qwen"   # generador común de C0, C1 y C2 (clave de MODELOS en C12-b)
    COMPARAR_MODELOS_EN_C2 = True
    C1_TAM_CHARS, C1_SOLAPE_CHARS = 1000, 200
    # C1 recibe tantos fragmentos como el MÁXIMO que entrega C2 (k=8 en C7).
    # Es deliberadamente favorable a C1: si C2 gana igual, el resultado es robusto.
    C1_K = 8

    # --- juez externo (RAGAS) ---
    JUEZ_URL     = "https://generativelanguage.googleapis.com/v1beta/openai/"
    JUEZ_MODELO  = "gemini-3.5-flash-lite"   # estable según ai.google.dev (7-nov-2026)
    JUEZ_EMB     = "gemini-embedding-001"
    JUEZ_SECRETO = "GEMINI_API_KEY"          # 🔑 panel izquierdo de Colab → Secretos
    JUEZ_PAUSA_S = 5.0                       # entre llamadas: el nivel gratuito permite 15 por minuto (11-nov-2026)

    # --- salidas (carpeta nueva en TU Drive: no toca la del equipo) ---
    # ← EDITAR: misma carpeta compartida que BANCO.
    DIR_RES = Path("/content/drive/MyDrive/resultados_evaluacion_tesis")

EVAL.DIR_RES.mkdir(parents=True, exist_ok=True)
assert EVAL.SPLIT in ("desarrollo", "prueba")
if EVAL.SPLIT == "prueba" and not EVAL.CONFIG_CONGELADA:
    raise SystemExit("⛔ SPLIT='prueba' exige CONFIG_CONGELADA=True. Calibra solo con 'desarrollo'.")
print(f"✅ Split: {EVAL.SPLIT} · resultados en {EVAL.DIR_RES}")

Mounted at /content/drive
✅ Split: prueba · resultados en /content/drive/MyDrive/resultados_evaluacion_tesis


In [3]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Material tesis - Ley Protección Datos")
destino = next((d for d in BASE.iterdir()
                if d.is_dir() and (d / "banco_etiquetado.csv").exists()), None)
assert destino, f"No encuentro banco_etiquetado.csv dentro de {BASE}"

EVAL.BANCO   = destino / "banco_etiquetado.csv"
EVAL.DIR_RES = destino

print("DIR_RES :", EVAL.DIR_RES)
for f in sorted(destino.glob("*")):
    print(f"   {f.stat().st_size:>10,} B   {f.name}")

faltan = [n for n in ("respuestas_desarrollo.jsonl", "ragas_desarrollo.jsonl")
          if not (destino / n).exists()]
print("\n", "✅ están los dos .jsonl" if not faltan else f"❌ faltan: {faltan}")

DIR_RES : /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis
      571,434 B   Evaluacion_RAG_C0_C1_C2.ipynb
        4,096 B   Leyes
          196 B   abstencion_corregida_prueba.csv
       42,778 B   auditoria_manual_prueba.csv
      135,092 B   banco_ampliado_500.csv
       25,101 B   banco_etiquetado.csv
       27,064 B   banco_piloto_100.csv
       13,679 B   banco_v11.csv
          299 B   metadatos_prueba.json
      380,518 B   metricas_auto_prueba.csv
        4,096 B   rag_indice
       28,844 B   ragas_desarrollo.jsonl
       19,535 B   ragas_prueba.jsonl
       12,810 B   recall_prueba.csv
    1,863,395 B   respuestas_desarrollo.jsonl
    1,470,201 B   respuestas_prueba.jsonl
          691 B   tabla_hipotesis_prueba.csv
          871 B   tabla_hipotesis_prueba.tex
          654 B   tabla_metricas_auto_prueba.csv
          843 B   tabla_metricas_auto_prueba.tex
          183 B   tabla_ragas_prueba.csv
          312 B   tabla_ragas_prueba.te

In [ ]:
# ═══════════════════════════════════════════════════════════════
# M · MUESTRA PILOTO ESTRATIFICADA DEL BANCO DE 500
# ═══════════════════════════════════════════════════════════════
# Autosuficiente: no depende de la celda de rutas. Solo necesita
# que el Drive esté montado (celda E1).
import pandas as pd
from pathlib import Path

N_POR_SPLIT = 50
SEMILLA     = 42
ARCHIVO     = "banco_ampliado_500.csv"

# ── Localizar la carpeta compartida ─────────────────────────────
# Tres intentos, del más barato al más caro. No se hace rglob sobre
# todo el Drive: con el acceso directo montado tarda minutos.
def hallar_carpeta():
    if "CARPETA" in globals() and (Path(globals()["CARPETA"]) / ARCHIVO).exists():
        return Path(globals()["CARPETA"])                      # ya la tenías
    try:
        if (Path(EVAL.DIR_RES) / ARCHIVO).exists():
            return Path(EVAL.DIR_RES)                          # la de la evaluación
    except Exception:
        pass
    raiz = Path("/content/drive/MyDrive")
    for d in sorted(raiz.glob("*")) + sorted(raiz.glob("*/*")):
        if d.is_dir() and (d / ARCHIVO).exists():
            return d                                           # búsqueda a dos niveles
    return None

CARPETA = hallar_carpeta()
if CARPETA is None:
    print(f"❌ No encontré {ARCHIVO} en el Drive.\n")
    print("Carpetas visibles en MyDrive:")
    for d in sorted(Path("/content/drive/MyDrive").glob("*")):
        if d.is_dir(): print("   ", d.name)
    raise SystemExit("Sube banco_ampliado_500.csv a la carpeta compartida y vuelve a correr M.")

print(f"📁 {CARPETA}")

# ── Muestreo estratificado ──────────────────────────────────────
# Estratificado y no aleatorio puro: con 100 al azar, multinorma y
# fuera_corpus pueden quedar en 2 casos y el piloto no diría nada
# sobre ellas. Semilla fija → reproducible.
DESTINO = CARPETA / f"banco_piloto_{N_POR_SPLIT*2}.csv"
B = pd.read_csv(CARPETA / ARCHIVO, dtype=str, keep_default_na=False)
assert len(B) == 500, f"Esperaba 500 filas, hay {len(B)}"

partes = []
for split, g in B.groupby("split", sort=True):
    prop = (g.tipo.value_counts(normalize=True) * N_POR_SPLIT).round().astype(int)
    prop = prop.clip(lower=3)
    while prop.sum() > N_POR_SPLIT: prop[prop.idxmax()] -= 1
    while prop.sum() < N_POR_SPLIT: prop[prop.idxmin()] += 1
    for tipo, n in prop.items():
        sub = g[g.tipo == tipo]
        partes.append(sub.sample(min(n, len(sub)), random_state=SEMILLA))

M = pd.concat(partes).sort_values("id").reset_index(drop=True)
M.to_csv(DESTINO, index=False)

EVAL.BANCO = DESTINO                   # ← la evaluación usa el piloto
EVAL.COMPARAR_MODELOS_EN_C2 = False    # solo el generador principal

print(f"\n💾 {DESTINO.name} · {len(M)} preguntas · semilla {SEMILLA}")
for s, g in M.groupby("split"):
    print(f"   {s:12}{len(g):>3}  {dict(g.tipo.value_counts())}")
resp = M[M.debe_abstenerse.str.lower() != "true"]
print(f"   respondibles {len(resp)} · confianza {dict(resp.confianza_etiqueta.value_counts())}")
print(f"\n✅ EVAL.BANCO → {EVAL.BANCO.name}")
print(f"   SPLIT = {EVAL.SPLIT}  ← debe decir 'desarrollo'")

📁 /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis

💾 banco_piloto_100.csv · 100 preguntas · semilla 42
   desarrollo   50  {'juridica': np.int64(26), 'coloquial': np.int64(17), 'multinorma': np.int64(4), 'fuera_corpus': np.int64(3)}
   prueba       50  {'juridica': np.int64(27), 'coloquial': np.int64(17), 'multinorma': np.int64(3), 'fuera_corpus': np.int64(3)}
   respondibles 94 · confianza {'alta': np.int64(85), 'media': np.int64(9)}

✅ EVAL.BANCO → banco_piloto_100.csv
   SPLIT = prueba  ← debe decir 'desarrollo'


In [4]:
if "CARPETA" not in globals():
    CARPETA = EVAL.DIR_RES          # v1.1: sin la celda M
# ═══════════════════════════════════════════════════════════════
# RUTAS · apuntar la evaluación a la carpeta compartida real
# ═══════════════════════════════════════════════════════════════
# Va DESPUÉS de M (que define CARPETA) y ANTES de E2.
# Sin esto, C2 del notebook del equipo busca en la carpeta por
# defecto con guiones bajos, que no existe.
from pathlib import Path

EVAL.DIR_RES = CARPETA
RUTA_LEYES   = CARPETA / "Leyes"          # por si C2 lee el global
if hasattr(EVAL, "RUTA_LEYES"):
    EVAL.RUTA_LEYES = RUTA_LEYES

pdfs = sorted(RUTA_LEYES.glob("*.pdf"))
print(f"📁 DIR_RES    → {EVAL.DIR_RES}")
print(f"📁 RUTA_LEYES → {RUTA_LEYES}")
print(f"📄 {len(pdfs)} PDF:")
for p in pdfs:
    print(f"   {p.name:<45} {p.stat().st_size/1e6:>6.1f} MB")
assert len(pdfs) == 6, f"Esperaba 6 PDF, hay {len(pdfs)}"
print(f"\n🎫 BANCO      → {EVAL.BANCO.name}")
print(f"🎫 SPLIT      → {EVAL.SPLIT}")

# ═══════════════════════════════════════════════════════════════
# V1.1 · CONFIGURACIÓN DE LA VERSIÓN 1.1  (split nuevo + parches P2, P3 y umbral)
# ═══════════════════════════════════════════════════════════════
# Orden v1.1: E1 → (celda que busca banco_etiquetado) → RUTAS (esta) → E2 → V3 → E3 → E4 → E5
#             → E6 → E6-bis → E7 → E8-bis → E8 → E9-pre → E9 → E10.   NO correr M ni DESHACER M.
import os
EVAL.V11 = True
EVAL.BANCO = CARPETA / "banco_v11.csv"          # 50 preguntas nuevas del banco de 500 (25 del abogado + 25 al azar)
EVAL.DIR_RES = CARPETA / "v11"; EVAL.DIR_RES.mkdir(exist_ok=True)
EVAL.SPLIT, EVAL.CONFIG_CONGELADA, EVAL.APLICAR_PARCHE_P1 = "prueba", True, False
EVAL.COMPARAR_MODELOS_EN_C2 = False             # solo el generador principal (Qwen3 8B)
UMBRAL_V11 = 0.005                              # calibrado en DESARROLLO (E11): recall 62,2 % → 68,9 %
os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"    # P3: en v1.0 la ventana por defecto (4.096) podía recortar prompts
assert EVAL.BANCO.exists(), f"Falta {EVAL.BANCO}: súbelo a la carpeta de resultados"

# P2 · tokenizador BM25 corregido (se inyecta en C6; conserva el de v1.0 como tokenizar_v10)
PARCHE_P2 = r'''
# ── PARCHE P2 (v1.1) · tokenizador BM25: números de 1 dígito, "7°"→"7", "21.719"→"21719", stemming ──
tokenizar_v10 = tokenizar
try:
    from nltk.stem.snowball import SpanishStemmer
    _STEM = SpanishStemmer().stem
except Exception:
    _STEM = lambda w: w
_P_MILES = re.compile(r"(?<!\d)(\d{1,3})\.(\d{3})(?!\d)")
_P_ORDINAL = re.compile(r"(\d)\s*[°º]")
def tokenizar(t):
    t = t.lower()
    t = _P_MILES.sub(r"\1\2", t)
    t = _P_ORDINAL.sub(r"\1", t)
    t = re.sub(r"[°º]", " ", t)
    t = unidecode(t)
    out = []
    for w in re.findall(r"[a-z0-9]+", t):
        if w.isdigit():
            out.append(w)
        elif len(w) > 1 and w not in STOPWORDS_ES:
            out.append(_STEM(w))
    return out
'''
print(f"\n🆕 V1.1 · banco {EVAL.BANCO.name} · resultados en {EVAL.DIR_RES} · umbral {UMBRAL_V11} · ctx 8192")

📁 DIR_RES    → /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis
📁 RUTA_LEYES → /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis/Leyes
📄 6 PDF:
   DTO-100_Constitucion_Politica_Republica.pdf      0.5 MB
   LEY-19628_Proteccion_Vida_Privada.pdf            0.1 MB
   LEY-20285_Acceso_Informacion_Publica.pdf         0.1 MB
   Ley-19496_Proteccion_Derechos_Consumidores.pdf    0.3 MB
   Ley-20416_Especial_Pymes.pdf                     0.1 MB
   Ley-21719_Proteccion_Datos.pdf                   0.2 MB

🎫 BANCO      → banco_etiquetado.csv
🎫 SPLIT      → prueba

🆕 V1.1 · banco banco_v11.csv · resultados en /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis/v11 · umbral 0.005 · ctx 8192


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DESHACER M · volver al banco original para la corrida de prueba
# ═══════════════════════════════════════════════════════════════
EVAL.BANCO = CARPETA / "banco_etiquetado.csv"
EVAL.COMPARAR_MODELOS_EN_C2 = True      # como en desarrollo: C2 con qwen y gemma

assert EVAL.BANCO.exists(), f"No encuentro {EVAL.BANCO}"
import pandas as pd
B = pd.read_csv(EVAL.BANCO, dtype=str, keep_default_na=False)
print(f"✅ BANCO → {EVAL.BANCO.name} ({len(B)} preguntas)")
print(f"   SPLIT = {EVAL.SPLIT}  ← debe decir 'prueba'")
print(f"   split 'prueba' en el banco: {len(B[B.split=='prueba'])} preguntas")
assert len(B) == 100, "⚠️ Este no es el banco de 100 — revisa el nombre del archivo"

✅ BANCO → banco_etiquetado.csv (100 preguntas)
   SPLIT = prueba  ← debe decir 'prueba'
   split 'prueba' en el banco: 50 preguntas


## E2 · Cargar el pipeline del equipo (sin copiar ni reescribir código)

Se clona el repositorio y se ejecutan las celdas de `RAG_Leyes_PYME.ipynb` identificadas por su etiqueta (`C3 ·`, `C5 ·`…). No se ejecutan `C5-bis` (sobrescribiría el respaldo compartido en Drive), `C11`, `C14`, `C15` ni `C16`: esta evaluación las reemplaza.

**Parche P1 (recomendado, opcional con `APLICAR_PARCHE_P1`).** Al reproducir C3–C4 con los PDF oficiales detectamos dos fallas de `P_ARTICULO`:

1. **Texto perdido.** Los artículos que la Ley 21.719 inserta en la 19.628 se escriben entre comillas (`"Artículo 3°.- Principios…`). El regex no los reconoce y ese texto queda fuera del corpus: **≈13.600 caracteres**, incluidos el art. 1° (objeto), el 1° bis (ámbito territorial), las modificaciones a las definiciones del art. 2° y el **art. 3° (principios)**. Lo mismo ocurre en encabezados entrecomillados de las leyes 20.285 y 20.416 (p. ej., el ámbito de aplicación del Estatuto PYME).
2. **Artículos fantasma.** Frases del decreto promulgatorio como "*Artículo 93 de la Constitución…*", partidas al inicio de línea, se indexan como artículos (8 chunks de ruido en la 21.719).

El parche **envuelve** el regex sin modificarlo: quita una comilla inicial antes de evaluar y descarta coincidencias cuyo texto empieza con "de/del" (referencias, no encabezados). Probado sobre los 6 PDF: recupera 10 artículos reales, elimina 3 fantasmas y no pierde ningún artículo legítimo. Limitación conocida: las modificaciones al art. 2° quedan citadas como "art. 1° bis" (el texto se recupera, la etiqueta es imperfecta).

⚠️ Con el parche, el corpus deja de tener 1.396 fragmentos: **el equipo debe decidir si lo adopta y actualizar la cifra del póster**.

In [4]:
# ═══════════════════════════════════════════════════════════════
# E2 · CARGAR EL PIPELINE DEL EQUIPO (recuperación: C0 → C10-bis)
# ═══════════════════════════════════════════════════════════════
import json, re, subprocess
from IPython import get_ipython

DIR_REPO = Path("/content/repo_tesis")
if not DIR_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", EVAL.REPO_URL, str(DIR_REPO)], check=True)
COMMIT = subprocess.run(["git", "-C", str(DIR_REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

def celdas_del_equipo(ruta):
    """{etiqueta: código} a partir del encabezado '# C5 · …' de cada celda."""
    nb = json.loads(Path(ruta).read_text(encoding="utf-8"))
    out = {}
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        cabeza = "\n".join(src.splitlines()[:3])
        m = re.search(r"#\s*(C\d+(?:-[a-z]+)?)\s*·", cabeza)
        if m:
            out[m.group(1)] = src
        elif "COHERENCIA" in cabeza:
            out["COHERENCIA"] = src
    return out

PARCHE_P1 = r"""
# ── PARCHE P1 (inyectado por el notebook de evaluación) ──
class _ParcheArticulo:
    def __init__(self, p):
        self.p, self.pattern = p, p.pattern
    def match(self, ln):
        s = ln.lstrip()
        if s[:1] in '"“”«':
            s = s[1:]
        m = self.p.match(s)
        if m and re.match(r"del?\s", (m.group(6) or "").lstrip()):
            return None
        return m
P_ARTICULO = _ParcheArticulo(P_ARTICULO)
"""

def preparar(etiqueta, src):
    """Únicas modificaciones al código del equipo, todas verificadas con assert."""
    if etiqueta == "C2" and EVAL.RUTA_LEYES:
        src, n = re.subn(r'^RUTA_LEYES = Path\(".*?"\)', f'RUTA_LEYES = Path("{EVAL.RUTA_LEYES}")', src, flags=re.M)
        assert n == 1, "No encontré la línea RUTA_LEYES en C2"
    if etiqueta == "C3" and EVAL.APLICAR_PARCHE_P1:
        marca = "def es_estructural(linea):"
        assert src.count(marca) == 1, "C3 cambió: revisa dónde inyectar el parche"
        src = src.replace(marca, PARCHE_P1 + "\n" + marca)
    if etiqueta == "C5":
        # C3–C4 se regeneran siempre, así que el índice se reconstruye para
        # garantizar que embeddings y chunks salgan del MISMO código.
        assert "REINDEXAR = False" in src, "C5 cambió: revisa la variable REINDEXAR"
        src = src.replace("REINDEXAR = False", "REINDEXAR = True")
    if getattr(EVAL, "V11", False):
        if etiqueta == "C1":
            src += f"\nCFG.UMBRAL_RERANK = {UMBRAL_V11}   # v1.1: calibrado en desarrollo\n"
        if etiqueta == "C6":
            marca = "# el corpus disperso usa el mismo texto contextual que el denso"
            assert src.count(marca) == 1, "C6 cambió: revisa dónde inyectar P2"
            src = src.replace(marca, PARCHE_P2 + "\n" + marca)
    return src

def correr(etiquetas):
    for et in etiquetas:
        assert et in CELDAS, f"Falta la celda {et} en el notebook del equipo"
        print(f"\n{'━'*70}\n▶ {et}\n{'━'*70}")
        r = get_ipython().run_cell(preparar(et, CELDAS[et]))
        if not r.success:
            raise RuntimeError(f"Falló la celda {et} del notebook del equipo — corrígela allá, no aquí.")

CELDAS = celdas_del_equipo(DIR_REPO / EVAL.NOTEBOOK)
print(f"Repositorio @ {COMMIT} · celdas encontradas: {sorted(CELDAS)}")

# C3 y C4 se corren siempre: así los chunks salen del código vigente y no de un
# respaldo de otra versión (la cifra de chunks se reporta abajo).
correr(["C0", "C1", "C2", "C3", "C4", "C5", "COHERENCIA", "C6", "C7",
        "C8", "C8-bis", "C9", "C10", "C10-bis"])

N_CHUNKS = len(CHUNKS)
print(f"\n✅ Pipeline cargado · {N_CHUNKS:,} chunks · {len(ARTICULOS_FULL):,} artículos · "
      f"parche P1 {'ACTIVO' if EVAL.APLICAR_PARCHE_P1 else 'inactivo'}")

Repositorio @ 3a68857 · celdas encontradas: ['C0', 'C1', 'C10', 'C10-bis', 'C10-ter', 'C11', 'C12-a', 'C12-b', 'C13', 'C14', 'C15', 'C16', 'C2', 'C3', 'C4', 'C4-bis', 'C5', 'C5-bis', 'C6', 'C7', 'C8', 'C8-bis', 'C9', 'COHERENCIA']

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Python : 3.13.16
GPU    : True Tesla T4

✅ Si viste errores de dependencias arriba: Runtime → Reiniciar sesión → vuelve a correr SOLO esta celda.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📁 Estructura creada en /content/rag_leyes
    /content/rag_leyes/pdf
    /content/rag_leyes/leyes_md
    /content/rag_leyes/chunks
    /content/rag_leyes/chroma

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📂 Origen  : /conte

,archivo,ley,paginas,chars,chars_por_pagina,articulos,boilerplate,notas_marginales
0,DTO-100_Constitucion_Politica_Republica.pdf,Dto 100 Constitucion Politica Republica,139,38965,3477,14,4,1753
1,LEY-19628_Proteccion_Vida_Privada.pdf,Ley 19628,13,32034,2884,27,5,36
2,LEY-20285_Acceso_Informacion_Publica.pdf,Ley 20285,22,70543,3639,68,5,2
3,Ley-19496_Proteccion_Derechos_Consumidores.pdf,Ley 19496,83,209213,3095,147,5,677
4,Ley-20416_Especial_Pymes.pdf,Ley 20416,25,78471,3568,59,4,2
5,Ley-21719_Proteccion_Datos.pdf,Ley 21719,56,157112,3233,90,4,34



📂 Markdown guardado en: /content/rag_leyes/leyes_md

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 1,439 chunks · 405 artículos
   🔗 649 bloques FUSIONADOS (enumeraciones y frases cortadas) — no es pérdida
   🗑️  61 titulillos DESCARTADOS por no tener contenido normativo

Chunks por ley:


tipo,articulo,inciso
ley,,
Dto 100 Constitucion Politica Republica,8,70
Ley 19496,79,377
Ley 19628,15,56
Ley 20285,40,156
Ley 20416,44,201
Ley 21719,77,316



Largo de chunk (chars):


,chars
count,1439.0
mean,699.0
std,709.0
min,35.0
25%,260.0
50%,451.0
75%,912.0
max,4000.0


... (8 outputs left out)

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

✅ matriz: (1439, 1024)
   indexados 1,000/1,439
   indexados 1,439/1,439

✅ ChromaDB lista — 1,439 vectores en /content/rag_leyes/chroma

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ COHERENCIA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ChromaDB : 1,439 vectores
  CHUNKS   : 1,439 chunks

✅ Índice y chunks coinciden exactamente.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C6
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ BM25 sobre 1,439 documentos
   vocabulario ≈ 3,105 términos
   ejemplo de tokenización: ['plaz', 'articul', '12']

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C7
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⏳ Vectorizando prototipos de ruta …
'¿Qué multa me arriesgo si se filtran los datos de mi…'
   → ruta: SEGURIDAD (sim 0.721) · pesos D/S 0.6/0.4

'¿Qué dice el artículo 12?…'
   → ruta: OBLIGACIONES 

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ FlagReranker · BAAI/bge-reranker-v2-m3
🔤 expandida: …supresión eliminación cancelación titular consumidor obligación deber del responsable
🔀 ruta=derechos · densa=25 dispersa=25 · solo-D 17 / ambos 8 / solo-S 17 → 42 candidatos

🎯 Top-8 tras reranker (ruta derechos):
  🟢 [F1] 0.817  Ley 21719, Artículo 7°, inciso 1
           Derecho de supresión. El titular de datos tiene derecho a solicitar y obtener del responsable, la eliminación …
  🟢 [F2] 0.712  Ley 19628, Artículo 6°, inciso 1
           Los datos personales deberán ser eliminados o cancelados cuando su almacenamiento carezca de fundamento legal …
  🟡 [F3] 0.618  Ley 21719, Artículo 14 bis
           Deber de secreto o confidencialidad.
El responsable de datos está obligado a mantener secreto o confidencialid…
  🟡 [F4] 0.486  Ley 19628, Artículo 12
           Toda persona tiene derecho a exigir a quien sea responsable de un banco, que se dedique en forma pública o pri…
  🔴 [F5] 0.292  Ley 21719, Artículo 14, inciso 1
         

In [5]:
# ═══════════════════════════════════════════════════════════════
# V3 · ¿el parche inyectado y el código fusionado dan el mismo corpus?
# ═══════════════════════════════════════════════════════════════
# Desarrollo se corrió con el parche P1 inyectado sobre el repo antiguo.
# Prueba se correrá con P1 ya fusionado en C3. La huella de configuración
# va a diferir porque hash_config() incluye el hash del commit, así que
# hay que DEMOSTRAR que el corpus resultante es idéntico.
#
# Córrela dos veces: con APLICAR_PARCHE_P1 en True y en False (volviendo
# a ejecutar E2 entremedio). Si la huella del corpus coincide, el parche
# es un no-op sobre el código fusionado.
import hashlib

def huella_corpus(chunks):
    """Hash del contenido, independiente del orden de inserción."""
    filas = sorted(f"{c['chunk_id']}\x1f{c['texto']}" for c in chunks)
    return hashlib.sha256("\x1e".join(filas).encode()).hexdigest()[:16]

print(f"APLICAR_PARCHE_P1 : {EVAL.APLICAR_PARCHE_P1}")
print(f"commit            : {COMMIT}")
print(f"fragmentos        : {len(CHUNKS):,}")
print(f"artículos         : {len({c['articulo_id'] for c in CHUNKS}):,}")
print(f"huella del corpus : {huella_corpus(CHUNKS)}")

faltan = [k for k in ("Artículo 1°", "Artículo 1° bis", "Artículo 3°")
          if k not in {c["articulo_label"] for c in CHUNKS
                       if "21719" in c["ley"].replace(".", "")}]
print("P1 ·", "✅ 1°, 1° bis y 3° presentes" if not faltan else f"🚨 FALTAN {faltan}")


APLICAR_PARCHE_P1 : False
commit            : 3a68857
fragmentos        : 1,439
artículos         : 404
huella del corpus : 6b67bcbf86797b89
P1 · ✅ 1°, 1° bis y 3° presentes


In [ ]:
print("articulo_id únicos:", df.articulo_id.nunique())

articulo_id únicos: 404


In [6]:
# ═══════════════════════════════════════════════════════════════
# E3 · UTILIDADES DE EVALUACIÓN (funciones puras, sin modelos)
# ═══════════════════════════════════════════════════════════════
import re, json, hashlib, unicodedata
import numpy as np, pandas as pd

# ---------- claves de artículo: "21719:7", "21719:14quáter", "cpr:19" ----------
def ley_de(nombre):
    m = re.search(r"(\d{2})\.?(\d{3})", nombre)
    return (m.group(1) + m.group(2)) if m else "cpr"

def norm_clave(c):
    c = unicodedata.normalize("NFC", str(c).strip().lower().replace(" ", ""))
    return re.sub(r"__r\d+$", "", c)

def clave_de_id(articulo_id):
    """'Ley-21719_Proteccion_Datos__art_14bis__r2' → '21719:14bis'.
    Se ignora el sufijo __rN: el banco identifica artículos, no apariciones."""
    stem, art = articulo_id.split("__art_", 1)
    return norm_clave(f"{ley_de(stem)}:{art}")

def claves_de_chunk(c):
    if "articulos" in c:                       # fragmento fijo de C1
        return set(c["articulos"])
    return {clave_de_id(c["articulo_id"])}

# ---------- C1: segmentación fija con mapa de artículos ----------
def mapa_articulos(md_texto, ley):
    """[(offset_inicio, clave)] usando el MISMO detector de encabezados que C2."""
    marcas, pos = [], 0
    for ln in md_texto.split("\n"):
        m = P_ARTICULO.match(ln.strip())
        if m:
            suf = (m.group(4) or "").lower() or (m.group(5) or "").upper()
            marcas.append((pos, norm_clave(f"{ley}:{m.group(2)}{suf}")))
        pos += len(ln) + 1
    return marcas

def segmentar_fijo(texto, tam, solape):
    """Ventanas de ~tam caracteres que cortan en el espacio más cercano,
    sin mirar la estructura de la ley (ese es justamente el punto de C1)."""
    spans, i, n = [], 0, len(texto)
    while i < n:
        f = min(n, i + tam)
        if f < n:
            esp = texto.rfind(" ", i + tam // 2, f)
            f = esp if esp > i else f
        spans.append((i, f))
        if f >= n:
            break
        i = max(f - solape, i + 1)
    return spans

def articulos_en(span, marcas):
    """Claves de artículo que el fragmento toca, ordenadas por cobertura."""
    ini, fin = span
    cubre = {}
    for j, (p, k) in enumerate(marcas):
        p_fin = marcas[j + 1][0] if j + 1 < len(marcas) else float("inf")
        inter = min(fin, p_fin) - max(ini, p)
        if inter > 0:
            cubre[k] = cubre.get(k, 0) + inter
    return [k for k, _ in sorted(cubre.items(), key=lambda x: -x[1])]

# ---------- referencias explícitas a artículos dentro de una respuesta ----------
SUF = r"(bis|ter|qu[áa]ter|quinquies|sexies|septies|octies|nonies|decies)"
P_REF = re.compile(
    r"\bart(?:[íi]culo|\.)?s?\s*(\d{1,3})\s*[°º]?\s*" + SUF + r"?"
    r"(?=(?P<cola>[^\n\[\]]{0,70}))", re.I)    # lookahead: no consume la cita siguiente
P_LEY_REF = re.compile(r"\bley\s*(?:n[°º.]?\s*)?(\d{2})\.?(\d{3})|\bconstituci[óo]n"
                       r"|(?P<otra>\bc[óo]digo\b|\bd\.?\s?f\.?\s?l\.?\b|\bdecreto\b|\breglamento\b)", re.I)

def referencias(texto):
    """[(clave, ley_conocida)] de menciones 'artículo 7° de la Ley 21.719'.
    Si la ley no se nombra en las 70 letras siguientes, la clave queda con ley '?'."""
    out = []
    for m in P_REF.finditer(texto or ""):
        num, suf = m.group(1), (m.group(2) or "").lower().replace("quater", "quáter")
        lm = P_LEY_REF.search(m.group("cola") or "")
        if lm and lm.group("otra"):
            ley = "externa"                     # Código del Trabajo, decretos, etc.
        elif lm:
            ley = (lm.group(1) + lm.group(2)) if lm.group(1) else "cpr"
        else:
            ley = "?"
        out.append(norm_clave(f"{ley}:{num}{suf}"))
    return out

def clasificar_referencias(refs, claves_corpus, leyes_corpus):
    """Separa referencias en: existentes, inexistentes (fabricadas) y no verificables
    (leyes fuera del corpus). Una referencia sin ley se acepta si el artículo
    existe en alguna de las normas del corpus (criterio conservador)."""
    ok, malas, externas = set(), set(), set()
    for r in refs:
        ley, art = r.split(":", 1)
        if ley == "?":
            hits = {f"{l}:{art}" for l in leyes_corpus if f"{l}:{art}" in claves_corpus}
            (ok.update(hits) if hits else malas.add(r))
        elif ley not in leyes_corpus:
            externas.add(r)
        else:
            (ok.add(r) if r in claves_corpus else malas.add(r))
    return ok, malas, externas

# ---------- abstención ----------
# "sin resultados relevantes": texto fijo que E7 devuelve cuando C2 no recupera fuentes
# (corrección 11-nov-2026: antes no se contaba como abstención).
P_ABST = re.compile(
    r"sin resultados relevantes|no cubre|no alcanza|no tengo cargad|no (?:est[áa]|se encuentra) (?:regulad|cubiert|contemplad)"
    r"|no (?:puedo|es posible) responder|fuera del (?:alcance|corpus)|no dispongo de (?:informaci|normativa)", re.I)
def se_abstuvo(respuesta):
    return bool(P_ABST.search(respuesta or ""))

# ---------- estadística ----------
def wilson(k, n, z=1.959964):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (c - h, c + h)

def pct(serie):
    """'72,5 % [60,1–82,3] (n=40)' para una serie booleana."""
    s = pd.Series(serie).dropna().astype(bool)
    n, k = len(s), int(s.sum())
    if n == 0:
        return "—"
    lo, hi = wilson(k, n)
    return f"{100*k/n:.1f} % [{100*lo:.1f}–{100*hi:.1f}] (n={n})".replace(".", ",")

def media_boot(serie, b=2000, semilla=42):
    s = pd.Series(serie).dropna().astype(float).values
    if len(s) == 0:
        return "—"
    rng = np.random.default_rng(semilla)
    bs = rng.choice(s, (b, len(s))).mean(axis=1)
    return (f"{100*s.mean():.1f} [{100*np.percentile(bs, 2.5):.1f}–"
            f"{100*np.percentile(bs, 97.5):.1f}] (n={len(s)})").replace(".", ",")

def hash_config():
    """Huella de todo lo calibrable. Cambia si alguien toca glosario, rutas,
    parámetros, prompt o el código del equipo."""
    cfg = {k: str(v) for k, v in vars(CFG).items() if not k.startswith("_")}
    rutas = {n: {k: v for k, v in r.items() if k != "centroide"} for n, r in RUTAS.items()}
    base = json.dumps([cfg, GLOSARIO_PYME, rutas, SYSTEM, COMMIT, EVAL.APLICAR_PARCHE_P1,
                       EVAL.C1_TAM_CHARS, EVAL.C1_SOLAPE_CHARS, EVAL.C1_K],
                      ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(base.encode()).hexdigest()[:12]

print("✅ Utilidades definidas")

✅ Utilidades definidas


In [7]:
# ═══════════════════════════════════════════════════════════════
# E4 · BANCO DE PREGUNTAS ETIQUETADO
# ═══════════════════════════════════════════════════════════════
if not EVAL.BANCO.exists():
    raise SystemExit(f"⛔ No encuentro {EVAL.BANCO}. Súbelo a esa ruta o cambia EVAL.BANCO.")

BANCO = pd.read_csv(EVAL.BANCO, dtype=str, keep_default_na=False)
faltan = {"id", "split", "tipo", "pregunta", "articulos_esperados",
          "debe_abstenerse", "respuesta_referencia"} - set(BANCO.columns)
assert not faltan, f"Columnas faltantes en el banco: {faltan}"
BANCO["debe_abstenerse"] = BANCO.debe_abstenerse.str.lower().isin(["true", "1", "si", "sí"])
BANCO["gold"] = BANCO.articulos_esperados.apply(
    lambda s: {norm_clave(x) for x in s.split("|") if x.strip()})
assert BANCO.id.is_unique, "Hay ids repetidos en el banco"

# --- ¿las etiquetas existen en el corpus que se acaba de construir? ---
CLAVES_CORPUS = {clave_de_id(c["articulo_id"]) for c in CHUNKS}
LEYES_CORPUS = {k.split(":")[0] for k in CLAVES_CORPUS}
inexistentes = sorted({k for g in BANCO.gold for k in g} - CLAVES_CORPUS)
if inexistentes:
    print(f"⚠️  {len(inexistentes)} artículos etiquetados NO existen en este corpus: {inexistentes}")
    print("   Sin el parche P1 es esperable para 21719:1, 21719:1bis y 21719:3 (texto perdido).")
    print("   Esas etiquetas cuentan como fallo de recuperación: es lo que hay que reportar.")

EVAL_SET = BANCO[BANCO.split == EVAL.SPLIT].reset_index(drop=True)
if getattr(EVAL, "PILOTO", 0):
    EVAL_SET = EVAL_SET.head(EVAL.PILOTO)
    print(f"🧪 MODO PILOTO: {len(EVAL_SET)} preguntas")
print(f"\n📋 Banco: {len(BANCO)} preguntas · split '{EVAL.SPLIT}': {len(EVAL_SET)}")
display(EVAL_SET.groupby("tipo").size().to_frame("n"))
print("Las etiquetas son PROPUESTAS a validar por el equipo (columna fuente_etiqueta).")


📋 Banco: 50 preguntas · split 'prueba': 50


,n
tipo,
coloquial,19
fuera_corpus,5
juridica,19
multinorma,7


Las etiquetas son PROPUESTAS a validar por el equipo (columna fuente_etiqueta).


In [8]:
# ═══════════════════════════════════════════════════════════════
# E5 · CONDICIÓN C1 (trozos fijos) + RECALL@K POR ETAPA  — sin LLM
# ═══════════════════════════════════════════════════════════════
# Se corre ANTES de cargar Ollama: C12-b mueve los embeddings a CPU.
import time

# ---------- índice C1: mismo texto limpio (leyes_md) y mismo encoder ----------
CHUNKS_C1 = []
for md_file in sorted(CFG.DIR_MD.glob("*.md")):
    texto = md_file.read_text(encoding="utf-8")
    ley_nombre = texto.split("\n")[0].lstrip("# ").strip()
    marcas = mapa_articulos(texto, ley_de(md_file.stem))
    for i, sp in enumerate(segmentar_fijo(texto, EVAL.C1_TAM_CHARS, EVAL.C1_SOLAPE_CHARS), 1):
        CHUNKS_C1.append(dict(chunk_id=f"C1__{md_file.stem}__{i}", ley=ley_nombre,
                              cita=f"{ley_nombre}, fragmento {i}",
                              texto=texto[sp[0]:sp[1]].strip(),
                              articulos=articulos_en(sp, marcas)))
POR_ID_C1 = {c["chunk_id"]: c for c in CHUNKS_C1}

try:
    client.delete_collection("c1_fijo")
except Exception:
    pass
COL_C1 = client.create_collection("c1_fijo", metadata={"hnsw:space": "cosine"})
print(f"⏳ Vectorizando {len(CHUNKS_C1):,} fragmentos fijos de C1 …")
EMB_C1 = ENC.docs([c["texto"] for c in CHUNKS_C1])
for i in range(0, len(CHUNKS_C1), 1000):
    lote = CHUNKS_C1[i:i+1000]
    COL_C1.add(ids=[c["chunk_id"] for c in lote], embeddings=EMB_C1[i:i+1000].tolist(),
               documents=[c["texto"] for c in lote])
print(f"✅ C1: {COL_C1.count():,} fragmentos de ~{EVAL.C1_TAM_CHARS} caracteres "
      f"(C2 tiene {N_CHUNKS:,} chunks jerárquicos)")

def recuperar_c1(pregunta, k=EVAL.C1_K):
    r = COL_C1.query(query_embeddings=[ENC.query(pregunta).tolist()], n_results=k,
                     include=["distances"])
    return [(POR_ID_C1[cid], 1 - d) for cid, d in zip(r["ids"][0], r["distances"][0])]

# ---------- etapas de C2, idénticas a construir_contexto() ----------
def etapas_c2(q):
    qe = expandir_consulta(q)
    densa = [POR_ID[c] for c, _ in buscar_densa(ENC.query(qe), CFG.K_DENSA)]
    disp = [POR_ID[c] for c, _ in buscar_dispersa(qe, CFG.K_DISPERSA)]
    fusion, _, _ = recuperar(qe, verbose=False)
    rrf10 = [POR_ID[c] for c, _, _ in fusion[:10] if c in POR_ID]
    finales, _ = construir_contexto(q, verbose=False)
    # ablación: el mismo C2 SIN expansión de vocabulario (aísla la brecha de lenguaje)
    fus0, _, p0 = recuperar(q, verbose=False)
    sin_exp = rerankear(q, fus0, top_k=p0["k"])
    return {"C2 · densa@25": densa, "C2 · BM25@25": disp, "C2 · RRF@10": rrf10,
            "C2 · final (top-k)": [c for c, _ in finales],
            "C2 sin expansión · final": [c for c, _ in sin_exp]}

def acierto(lista, gold):
    """(hit, rango_reciproco) de la primera posición con un artículo esperado."""
    for pos, c in enumerate(lista, 1):
        if claves_de_chunk(c) & gold:
            return True, 1 / pos
    return False, 0.0

filas = []
evaluables = EVAL_SET[EVAL_SET.gold.str.len() > 0]
t0 = time.time()
for r in evaluables.itertuples():
    etapas = etapas_c2(r.pregunta)
    etapas["C1 · densa fija (top-k)"] = [c for c, _ in recuperar_c1(r.pregunta)]
    for etapa, lista in etapas.items():
        hit, rr = acierto(lista, r.gold)
        filas.append(dict(id=r.id, tipo=r.tipo, etapa=etapa, hit=hit, rr=rr, k=len(lista)))
DF_RECALL = pd.DataFrame(filas)
DF_RECALL.to_csv(EVAL.DIR_RES / f"recall_{EVAL.SPLIT}.csv", index=False)
print(f"⏱️ {time.time()-t0:.0f}s para {len(evaluables)} preguntas")

orden = ["C2 · densa@25", "C2 · BM25@25", "C2 · RRF@10", "C2 · final (top-k)",
         "C2 sin expansión · final", "C1 · densa fija (top-k)"]
T_RECALL = (DF_RECALL.groupby("etapa")
            .agg(recall=("hit", pct), MRR=("rr", lambda s: f"{s.mean():.3f}".replace(".", ",")),
                 k_medio=("k", "mean"))
            .reindex(orden).round(1))
print("\n" + "═"*72 + f"\nRECALL@k POR ETAPA · split {EVAL.SPLIT}\n" + "═"*72)
display(T_RECALL)
print("\nPor tipo de pregunta (etapa final):")
display(DF_RECALL[DF_RECALL.etapa.isin(["C2 · final (top-k)", "C1 · densa fija (top-k)"])]
        .pivot_table(index="tipo", columns="etapa", values="hit", aggfunc=pct))
print("""
Lectura: 'C2 · final' < 'C2 · RRF@10' → el reordenador o el umbral descartan lo correcto.
         'C2 sin expansión' < 'C2 · final' → cuánto aporta el glosario (brecha de lenguaje, H3).
         'C1' vs 'C2 · final' → H2 a nivel de recuperación.""")

⏳ Vectorizando 740 fragmentos fijos de C1 …


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

✅ C1: 740 fragmentos de ~1000 caracteres (C2 tiene 1,439 chunks jerárquicos)
⏱️ 185s para 45 preguntas

════════════════════════════════════════════════════════════════════════
RECALL@k POR ETAPA · split prueba
════════════════════════════════════════════════════════════════════════


,recall,MRR,k_medio
etapa,,,
C2 · densa@25,"97,8 % [88,4–99,6] (n=45)","0,800",25.0
C2 · BM25@25,"88,9 % [76,5–95,2] (n=45)","0,596",25.0
C2 · RRF@10,"95,6 % [85,2–98,8] (n=45)","0,740",10.0
C2 · final (top-k),"95,6 % [85,2–98,8] (n=45)","0,836",7.2
C2 sin expansión · final,"91,1 % [79,3–96,5] (n=45)","0,831",7.1
C1 · densa fija (top-k),"86,7 % [73,8–93,7] (n=45)","0,705",8.0



Por tipo de pregunta (etapa final):


etapa,C1 · densa fija (top-k),C2 · final (top-k)
tipo,,
coloquial,"68,4 % [46,0–84,6] (n=19)","89,5 % [68,6–97,1] (n=19)"
juridica,"100,0 % [83,2–100,0] (n=19)","100,0 % [83,2–100,0] (n=19)"
multinorma,"100,0 % [64,6–100,0] (n=7)","100,0 % [64,6–100,0] (n=7)"



Lectura: 'C2 · final' < 'C2 · RRF@10' → el reordenador o el umbral descartan lo correcto.
         'C2 sin expansión' < 'C2 · final' → cuánto aporta el glosario (brecha de lenguaje, H3).
         'C1' vs 'C2 · final' → H2 a nivel de recuperación.


## E6 · Motor generativo local

Ejecuta C12-a, C12-b, C13 y C10-ter del equipo, sin cambios. C12-b descarga los tres modelos y mueve los embeddings a CPU para liberar VRAM.

In [9]:
# ═══════════════════════════════════════════════════════════════
# E6 · OLLAMA + GENERACIÓN (C12-a → C13, C10-ter)
# ═══════════════════════════════════════════════════════════════
correr(["C12-a", "C12-b", "C13", "C10-ter"])

# ── PARCHE P2 · generación con Ollama actual (verificado 8-nov-2026) ──
# Con Ollama actual, el sufijo "/no_think" de C13 ya NO apaga el razonamiento
# de Qwen3: el modelo gasta los 1.600 tokens pensando y la respuesta llega
# VACÍA. La documentación de Ollama indica reasoning_effort="none" en el
# endpoint compatible con OpenAI. DeepSeek-R1 no permite apagarlo: más tokens.
import subprocess
if "qwen3:8b" not in MODELOS.values():              # C12-b pudo caer a qwen3:4b
    r = subprocess.run(["ollama", "pull", "qwen3:8b"], capture_output=True, text=True)
    if r.returncode == 0:
        MODELOS["qwen"] = "qwen3:8b"
    else:
        print("⚠️  No se pudo descargar qwen3:8b:", (r.stderr or "")[-200:])

def llamar_llm(system, user, modelo=None):
    modelo = modelo or MODELOS.get("qwen")
    p = perfil_de(modelo)
    extra = {"reasoning_effort": "none"} if "qwen" in modelo else {}
    tope = 6000 if "deepseek" in modelo else 1600
    r = CLIENTE.chat.completions.create(
        model=modelo, temperature=p["temperatura"], max_tokens=tope,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user + p["sufijo"]}], **extra)
    return r.choices[0].message.content or ""

# v1.1: DeepSeek no se prueba (está excluido y su carga agotaba la RAM del entorno)
for m in [v for v in MODELOS.values() if "deepseek" not in v]:   # prueba: nadie responde vacío
    t, _ = separar_razonamiento(llamar_llm("Responde en una frase.", "¿Qué es un dato personal?", m))
    assert t.strip(), f"{m} devolvió una respuesta vacía: revisa Ollama antes de seguir"
    print(f"   {m}: OK ({len(t)} caracteres)")
    subprocess.run(["ollama", "stop", m], capture_output=True)   # libera memoria
# Advertencia: interrumpir una celda en Colab puede detener el servidor Ollama.
# Si aparece 'Connection refused', ejecuta correr(["C12-b"]) y vuelve a esta celda.

GENERADOR = MODELOS[EVAL.GENERADOR_FAMILIA]
print(f"\n✅ Generador común C0/C1/C2: {GENERADOR} · modelos en C2: {list(MODELOS.values())}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ C12-a
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ Instalando zstd…
  zstd: *** Zstandard CLI (64-bit) v1.5.5, by Yann Collet ***

▶ Instalando Ollama…
  rc = 0
  ##          90.1%
  #################################################################         91.5%
  ##################################################################        92.9%
  ###################################################################       94.3%
  #####################################################################     96.3%
  ######################################################################    97.6%
  ######################################################################    98.3%
  #######################################################################   99.0%
  #######################################################################   99.6%
  ###########################################################

In [10]:
MODELOS.pop("deepseek", None)
print("✅ modelos en C2:", MODELOS)

✅ modelos en C2: {'gemma': 'gemma4:e2b', 'qwen': 'qwen3:8b'}


In [11]:
# ═══════════════════════════════════════════════════════════════
# E6-bis · PARCHE P3 · un solo modelo cargado a la vez
# ═══════════════════════════════════════════════════════════════
# Con los tres modelos residentes, la RAM del entorno T4 se agotaba a
# mitad de la corrida: Ollama no descarga el anterior al pedirle otro.
# _llamar_base guarda la función original, así re-ejecutar esta celda
# no encadena el envoltorio consigo mismo.
# PARCHE P3 · un solo modelo cargado a la vez (evita agotar la RAM)
import gc, subprocess
_llamar_base = globals().get("_llamar_base", llamar_llm)
_ULTIMO = [None]
def llamar_llm(system, user, modelo=None):
    modelo = modelo or GENERADOR
    if _ULTIMO[0] and _ULTIMO[0] != modelo:
        subprocess.run(["ollama", "stop", _ULTIMO[0]], capture_output=True)
        gc.collect()
    _ULTIMO[0] = modelo
    return _llamar_base(system, user, modelo)
for m in MODELOS.values():
    subprocess.run(["ollama", "stop", m], capture_output=True)
gc.collect()
import psutil; print(f"P3 activo · RAM total {psutil.virtual_memory().total/1e9:.0f} GB · usada {psutil.virtual_memory().percent}%")


# ── V1.1 · P3: registrar cuántos tokens de entrada leyó el modelo en cada llamada ──
ULTIMO_USO = [None]
if not getattr(CLIENTE.chat.completions, "_v11", False):
    _create_base = CLIENTE.chat.completions.create
    def _create_v11(*a, **k):
        r = _create_base(*a, **k)
        ULTIMO_USO[0] = getattr(r, "usage", None)
        return r
    CLIENTE.chat.completions.create = _create_v11
    CLIENTE.chat.completions._v11 = True
print("P3 · registro de tokens activo · OLLAMA_CONTEXT_LENGTH =", os.environ.get("OLLAMA_CONTEXT_LENGTH"))

P3 activo · RAM total 14 GB · usada 48.7%
P3 · registro de tokens activo · OLLAMA_CONTEXT_LENGTH = 8192


In [12]:
MODELOS = {k: v for k, v in MODELOS.items() if 'deepseek' not in v}; print('Modelos en C2 (deepseek excluido por latencia):', list(MODELOS.values()))

Modelos en C2 (deepseek excluido por latencia): ['gemma4:e2b', 'qwen3:8b']


In [13]:
# ═══════════════════════════════════════════════════════════════
# E7 · CORRIDA DE GENERACIÓN C0 / C1 / C2  (se reanuda sola)
# ═══════════════════════════════════════════════════════════════
import time, shutil, datetime

# C0: mismas reglas de formato, tono y abstención que SYSTEM, pero sin fuentes.
# Se le pide citar "Ley, artículo" en texto: es la única forma de auditar sus citas.
SYSTEM_C0 = """Eres un asesor legal que le explica normativa chilena de protección de datos a dueños de PYMES sin formación jurídica.

REGLAS — no negociables
R1. Responde con tu conocimiento de la legislación chilena vigente.
R2. Toda afirmación jurídica indica su fuente en la frase con ley y artículo, por ejemplo: (Ley 21.719, artículo 7°).
R3. Si no sabes o el tema no es de protección de datos, dilo explícitamente: "La normativa que tengo cargada no cubre este punto", y señala qué norma habría que consultar. NO completes el vacío.
R4. Nunca inventes números de artículo, plazos, montos, porcentajes ni sanciones.

FORMATO DE RESPUESTA
**En corto:** 2-3 líneas, sin tecnicismos.
**Qué dice la ley:** viñetas, cada una con su ley y artículo.
**Qué significa para tu empresa:** traducción accionable.
**Ojo con esto:** solo si hay un plazo fatal, una multa o un riesgo concreto.

CIERRE OBLIGATORIO
Termina siempre con: "_Orientación general basada en la normativa cargada — no reemplaza asesoría legal para tu caso puntual._"
"""

def _generar(system, user, modelo):
    bruto = llamar_llm(system, user, modelo)
    return separar_razonamiento(bruto)

def generar(condicion, pregunta, modelo):
    """Devuelve la respuesta y, por fuente, las claves de artículo que contiene."""
    t0 = time.time()
    if condicion == "C0":
        limpio, pensado = _generar(SYSTEM_C0, f"PREGUNTA DEL CLIENTE PYME: {pregunta}", modelo)
        return dict(respuesta=limpio, razonamiento=pensado, contextos=[], fuentes=[],
                    ruta="", segundos=round(time.time() - t0, 2))
    if condicion == "C1":
        chunks, ruta = recuperar_c1(pregunta), "sin ruteo"
    else:  # C2: exactamente lo que hace consultar() de C13
        chunks, ruta = construir_contexto(pregunta, verbose=False)
        if not chunks:
            return dict(respuesta="Sin resultados relevantes.", razonamiento="", contextos=[],
                        fuentes=[], ruta=ruta, segundos=round(time.time() - t0, 2))
    limpio, pensado = _generar(SYSTEM, construir_prompt(pregunta, chunks, ruta), modelo)
    return dict(respuesta=limpio, razonamiento=pensado, ruta=ruta,
                contextos=[c.get("texto_expandido", c["texto"]) for c, _ in chunks],
                fuentes=[dict(marcador=f"F{i}", cita=c["cita"], claves=sorted(claves_de_chunk(c)))
                         for i, (c, _) in enumerate(chunks, 1)],
                segundos=round(time.time() - t0, 2))

# ---------- plan de corrida ----------
# ── V1.1: condición pareada C2v10 = C2 con la recuperación de la v1.0 (BM25 y umbral antiguos) ──
from contextlib import contextmanager
from rank_bm25 import BM25Okapi
if getattr(EVAL, "V11", False) and "BM25_V10" not in globals():
    BM25_V10 = BM25Okapi([tokenizar_v10(c["texto_embed"]) for c in CHUNKS])
@contextmanager
def _config_v10():
    g = globals(); viejos = (g["BM25"], g["tokenizar"], CFG.UMBRAL_RERANK)
    g["BM25"], g["tokenizar"], CFG.UMBRAL_RERANK = BM25_V10, tokenizar_v10, 0.05
    try:
        yield
    finally:
        g["BM25"], g["tokenizar"], CFG.UMBRAL_RERANK = viejos
_generar_v11 = generar
def generar(condicion, pregunta, modelo):
    if condicion == "C2v10":
        with _config_v10():
            return _generar_v11("C2", pregunta, modelo)
    return _generar_v11(condicion, pregunta, modelo)

PLAN = [("C0", GENERADOR), ("C1", GENERADOR), ("C2", GENERADOR)]
if getattr(EVAL, "V11", False):
    PLAN.append(("C2v10", GENERADOR))
if EVAL.COMPARAR_MODELOS_EN_C2:
    PLAN += [("C2", m) for m in MODELOS.values() if m != GENERADOR]

HUELLA = hash_config()
LOCAL = Path(f"/content/respuestas_{EVAL.SPLIT}.jsonl")
DRIVE = EVAL.DIR_RES / LOCAL.name
if DRIVE.exists() and not LOCAL.exists():
    shutil.copy(DRIVE, LOCAL)                      # reanudación tras desconexión

def leer_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")] if Path(p).exists() else []

hechas = {(f["condicion"], f["modelo"], f["id"]) for f in leer_jsonl(LOCAL) if not f.get("error")}
huellas_previas = {f["huella"] for f in leer_jsonl(LOCAL)}
if huellas_previas - {HUELLA}:
    print(f"⚠️  Hay filas generadas con otra configuración: {huellas_previas - {HUELLA}}.")
    print("   Mezclarlas invalida la comparación. Usa otra carpeta o borra el archivo.")
    if EVAL.SPLIT == "prueba":
        raise SystemExit("⛔ En 'prueba' no se mezclan configuraciones.")

pendientes = [(c, m, r) for c, m in PLAN for r in EVAL_SET.itertuples()
              if (c, m, r.id) not in hechas]
print(f"Huella de configuración: {HUELLA} · plan: {len(PLAN)} combinaciones × "
      f"{len(EVAL_SET)} preguntas · pendientes: {len(pendientes)}")

for j, (cond, modelo, r) in enumerate(pendientes, 1):
    fila = dict(id=r.id, split=EVAL.SPLIT, tipo=r.tipo, condicion=cond, modelo=modelo,
                pregunta=r.pregunta, huella=HUELLA, commit=COMMIT,
                fecha=datetime.datetime.now().isoformat(timespec="seconds"))
    if "ULTIMO_USO" in globals(): ULTIMO_USO[0] = None
    try:
        fila.update(generar(cond, r.pregunta, modelo))
        fila["prompt_tokens"] = getattr(globals().get("ULTIMO_USO", [None])[0], "prompt_tokens", None)
        if not str(fila.get("respuesta", "")).strip():
            fila["error"] = "respuesta vacía"          # nunca cuenta como hecha
    except Exception as e:
        fila["error"] = f"{type(e).__name__}: {e}"
    with open(LOCAL, "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")
    shutil.copy(LOCAL, DRIVE)                      # copia cerrada, nunca escritura parcial
    estado = "❌ " + fila["error"][:60] if fila.get("error") else f"{fila['segundos']:.0f}s"
    print(f"[{j}/{len(pendientes)}] {cond} · {modelo:16} · {r.id} · {estado}")

print("\n✅ Corrida completa (o reanudable): vuelve a ejecutar la celda si hubo errores.")

Huella de configuración: cd627464444c · plan: 4 combinaciones × 50 preguntas · pendientes: 88
[1/88] C2 · qwen3:8b         · G103 · 44s
[2/88] C2 · qwen3:8b         · G111 · 16s
[3/88] C2 · qwen3:8b         · G127 · 17s
[4/88] C2 · qwen3:8b         · G133 · 15s
[5/88] C2 · qwen3:8b         · G144 · 17s
[6/88] C2 · qwen3:8b         · G167 · 16s
[7/88] C2 · qwen3:8b         · G184 · 22s
[8/88] C2 · qwen3:8b         · G203 · 24s
[9/88] C2 · qwen3:8b         · G211 · 17s
[10/88] C2 · qwen3:8b         · G228 · 16s
[11/88] C2 · qwen3:8b         · G253 · 43s
[12/88] C2 · qwen3:8b         · G261 · 21s
[13/88] C2 · qwen3:8b         · G268 · 20s
[14/88] C2 · qwen3:8b         · G281 · 47s
[15/88] C2 · qwen3:8b         · G296 · 18s
[16/88] C2 · qwen3:8b         · G305 · 16s
[17/88] C2 · qwen3:8b         · G307 · 13s
[18/88] C2 · qwen3:8b         · G328 · 18s
[19/88] C2 · qwen3:8b         · G333 · 16s
[20/88] C2 · qwen3:8b         · G343 · 17s
[21/88] C2 · qwen3:8b         · G361 · 15s
[22/88] C2 ·

In [14]:
# ═══════════════════════════════════════════════════════════════
# E8-bis · PARCHE · parser de referencias legales y detector de citas
# ═══════════════════════════════════════════════════════════════
# Tres correcciones de MEDICIÓN (no tocan el agente, solo cómo se mide):
#  1. referencias(): extrae "artículo 7° de la Ley 21.719" y lo normaliza
#     a una clave ley:artículo. Sin ley identificable queda '?' y no
#     cuenta ni como acierto ni como invención.
#  2. verificar_citas(): suma los marcadores [F#] que el detector base
#     no capturaba.
#  3. leer_jsonl(): excluye deepseek-r1. La exclusión es por VIABILIDAD
#     (206 s por respuesta frente a 13-15 s), no por sus puntajes.
#     Debe quedar declarada así en el capítulo.
SUF = r"(bis|ter|qu[áa]ter|quinquies|sexies|septies|octies|nonies|decies)"
P_REF = re.compile(
    r"\bart(?:[íi]culo|\.)?s?\s*(\d{1,3})\s*[°º]?\s*" + SUF + r"?"
    r"(?=(?P<cola>[^\n\[\]]{0,70}))", re.I)    # lookahead: no consume la cita siguiente
P_LEY_REF = re.compile(r"\bley\s*(?:n[°º.]?\s*)?(\d{2})\.?(\d{3})|\bconstituci[óo]n"
                       r"|(?P<otra>\bc[óo]digo\b|\bd\.?\s?f\.?\s?l\.?\b|\bdecreto\b|\breglamento\b)", re.I)

def referencias(texto):
    """[(clave, ley_conocida)] de menciones 'artículo 7° de la Ley 21.719'.
    Si la ley no se nombra en las 70 letras siguientes, la clave queda con ley '?'."""
    out = []
    for m in P_REF.finditer(texto or ""):
        num, suf = m.group(1), (m.group(2) or "").lower().replace("quater", "quáter")
        # 1º ley inmediatamente ANTES: "(Ley 21.719, artículo 7°)"
        ventana = texto[max(0, m.start() - 30):m.start()]
        antes = [x for x in P_LEY_REF.finditer(ventana)
                 if re.fullmatch(r"[\s,;:(]*", ventana[x.end():])]   # pegada al artículo
        lm = antes[-1] if antes else None
        # 2º ley inmediatamente DESPUÉS: "artículo 7° de la Ley 21.719" (no la de la cita siguiente)
        if not lm:
            lm = P_LEY_REF.search(m.group("cola") or "")
            if lm and lm.start() > 30:
                lm = None
        if lm and lm.group("otra"):
            ley = "externa"                     # Código del Trabajo, decretos, etc.
        elif lm:
            ley = (lm.group(1) + lm.group(2)) if lm.group(1) else "cpr"
        else:
            ley = "?"
        out.append(norm_clave(f"{ley}:{num}{suf}"))
    return out


_vc_base = globals().get("_vc_base", verificar_citas)
def verificar_citas(texto, n):
    a = _vc_base(texto, n)
    a["usadas"] = sorted(set(a["usadas"]) | {int(x) for x in re.findall(r"\[F(\d+)\b", texto or "")})
    return a
_leer_base = globals().get("_leer_base", leer_jsonl)
def leer_jsonl(p):
    return [f for f in _leer_base(p) if "deepseek" not in f.get("modelo", "")]
print("Parser de citas actualizado · deepseek excluido del análisis")


Parser de citas actualizado · deepseek excluido del análisis


In [15]:
# ═══════════════════════════════════════════════════════════════
# E8 · MÉTRICAS AUTOMÁTICAS (sin juez): citas, acierto, abstención
# ═══════════════════════════════════════════════════════════════
RESP = pd.DataFrame([f for f in leer_jsonl(LOCAL) if not f.get("error")])
RESP = RESP.merge(BANCO[["id", "gold", "debe_abstenerse"]], on="id", how="left")
CLAVES_OFICIALES = CLAVES_CORPUS
if not EVAL.APLICAR_PARCHE_P1:
    print("⚠️  Sin parche P1 el corpus no contiene los arts. 1°, 1° bis y 3° de la 21.719:")
    print("   una cita correcta a ellos se contaría como inventada. Interpreta con cuidado.")

def medir(f):
    refs = referencias(f.respuesta)
    ok, malas, externas = clasificar_referencias(refs, CLAVES_OFICIALES, LEYES_CORPUS)
    aud = verificar_citas(f.respuesta, len(f.fuentes)) if f.fuentes else dict(usadas=[], invalidas=[])
    citadas = set(ok)
    for n in aud["usadas"]:
        if 1 <= n <= len(f.fuentes):
            citadas |= set(f.fuentes[n - 1]["claves"])
    pins = verificar_pinpoints(f.respuesta, f.contextos) if f.contextos else []
    abst = se_abstuvo(f.respuesta)
    return pd.Series(dict(
        n_refs=len(refs), refs_inexistentes="|".join(sorted(malas)),
        refs_externas="|".join(sorted(externas)),
        cita_inventada=bool(malas) or bool(aud["invalidas"]),
        cita_esperada=(bool(citadas & f.gold) if (f.gold and not f.debe_abstenerse) else np.nan),
        abstuvo=abst,
        abstencion_correcta=(abst if f.debe_abstenerse else np.nan),
        falsa_abstencion=(abst if not f.debe_abstenerse else np.nan),
        pinpoints_sospechosos=sum(p["estado"] != "ok" for p in pins),
    ))

MET = pd.concat([RESP, RESP.apply(medir, axis=1)], axis=1)
MET.drop(columns=["gold", "contextos"]).to_csv(EVAL.DIR_RES / f"metricas_auto_{EVAL.SPLIT}.csv", index=False)

T_AUTO = MET.groupby(["condicion", "modelo"]).agg(
    cita_inventada=("cita_inventada", pct),
    cita_articulo_esperado=("cita_esperada", pct),
    abstencion_correcta=("abstencion_correcta", pct),
    falsa_abstencion=("falsa_abstencion", pct),
    latencia_mediana_s=("segundos", "median"),
)
print("═"*72 + f"\nMÉTRICAS AUTOMÁTICAS · split {EVAL.SPLIT} · % [IC 95 % Wilson]\n" + "═"*72)
display(T_AUTO)
print("""
cita_inventada          → % de respuestas con ≥1 artículo inexistente o marcador [F#] inválido.
cita_articulo_esperado  → % de respuestas que citan al menos un artículo del banco (proxy de acierto).
abstencion_correcta     → % de preguntas fuera del corpus en que el sistema se abstuvo.
falsa_abstencion        → % de preguntas respondibles en que se abstuvo sin necesidad.
Criterio conservador: en C1 un fragmento citado cuenta con TODOS los artículos que toca.""")

# Hoja para auditoría humana: el detector de abstención y el parser de citas
# son reglas; una muestra revisada a mano permite reportar su precisión.
muestra = MET.sample(min(30, len(MET)), random_state=42)[
    ["id", "condicion", "modelo", "pregunta", "respuesta", "abstuvo", "refs_inexistentes", "cita_esperada"]]
muestra.assign(revisor_abstuvo="", revisor_refs_ok="").to_csv(
    EVAL.DIR_RES / f"auditoria_manual_{EVAL.SPLIT}.csv", index=False)
print(f"📝 Muestra para revisión manual: {EVAL.DIR_RES / f'auditoria_manual_{EVAL.SPLIT}.csv'}")

⚠️  Sin parche P1 el corpus no contiene los arts. 1°, 1° bis y 3° de la 21.719:
   una cita correcta a ellos se contaría como inventada. Interpreta con cuidado.
════════════════════════════════════════════════════════════════════════
MÉTRICAS AUTOMÁTICAS · split prueba · % [IC 95 % Wilson]
════════════════════════════════════════════════════════════════════════


,,cita_inventada,cita_articulo_esperado,abstencion_correcta,falsa_abstencion,latencia_mediana_s
condicion,modelo,,,,,
C0,qwen3:8b,"22,0 % [12,8–35,2] (n=50)","6,7 % [2,3–17,9] (n=45)","0,0 % [0,0–43,4] (n=5)","0,0 % [0,0–7,9] (n=45)",9.37
C1,qwen3:8b,"2,0 % [0,4–10,5] (n=50)","84,4 % [71,2–92,3] (n=45)","0,0 % [0,0–43,4] (n=5)","0,0 % [0,0–7,9] (n=45)",14.16
C2,qwen3:8b,"0,0 % [0,0–7,1] (n=50)","91,1 % [79,3–96,5] (n=45)","40,0 % [11,8–76,9] (n=5)","0,0 % [0,0–7,9] (n=45)",16.44
C2v10,qwen3:8b,"0,0 % [0,0–7,1] (n=50)","91,1 % [79,3–96,5] (n=45)","60,0 % [23,1–88,2] (n=5)","0,0 % [0,0–7,9] (n=45)",16.48



cita_inventada          → % de respuestas con ≥1 artículo inexistente o marcador [F#] inválido.
cita_articulo_esperado  → % de respuestas que citan al menos un artículo del banco (proxy de acierto).
abstencion_correcta     → % de preguntas fuera del corpus en que el sistema se abstuvo.
falsa_abstencion        → % de preguntas respondibles en que se abstuvo sin necesidad.
Criterio conservador: en C1 un fragmento citado cuenta con TODOS los artículos que toca.
📝 Muestra para revisión manual: /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis/v11/auditoria_manual_prueba.csv


In [ ]:
import json
RUTA = EVAL.DIR_RES / "respuestas_prueba.jsonl"
R = [json.loads(l) for l in open(RUTA, encoding="utf-8")]
print("campos:", list(R[0].keys()), "\n")

OJO = {"F01","F02","F03","F08","F10","Q17","Q32"}
for r in R:
    if r.get("condicion") == "C2" and r.get("id") in OJO:
        resp = (r.get("respuesta") or r.get("texto") or "")[:150]
        print(f"{r['id']} · {r.get('modelo','?'):12} · fuentes={len(r.get('contextos', r.get('fuentes', [])) )}")
        print(f"   {resp!r}\n")

campos: ['id', 'split', 'tipo', 'condicion', 'modelo', 'pregunta', 'huella', 'commit', 'fecha', 'respuesta', 'razonamiento', 'contextos', 'fuentes', 'ruta', 'segundos'] 

Q17 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

Q32 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

F01 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

F02 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

F03 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

F08 · qwen3:8b     · fuentes=0
   'Sin resultados relevantes.'

F10 · qwen3:8b     · fuentes=8
   '**RESPUESTA:**\n\nNo, no puedes usar en tu web fotos que encontraste en Google sin pagar derechos de autor, salvo que cumas con alguna de las excepcione'

Q17 · gemma4:e2b   · fuentes=0
   'Sin resultados relevantes.'

Q32 · gemma4:e2b   · fuentes=0
   'Sin resultados relevantes.'

F01 · gemma4:e2b   · fuentes=0
   'Sin resultados relevantes.'

F02 · gemma4:e2b   · fuentes=0
   'Sin resultados relevantes.'


In [ ]:
!ls -la "/content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis/" | grep abst

-rw-------  1 root root     196 Nov 10 12:41 abstencion_corregida_prueba.csv


In [6]:
# ═══════════════════════════════════════════════════════════════
# E9-pre · Configuración del juez de RAGAS
# ═══════════════════════════════════════════════════════════════
# max_tokens=8192: con el valor por defecto el juez truncaba la salida
# estructurada en las respuestas largas y la métrica fallaba.
# JUEZ_PAUSA_S=6: espaciado para no chocar con el límite por minuto.
import ragas.llms as _rl, functools
_rl.llm_factory = functools.partial(getattr(_rl, "_lf_base", _rl.llm_factory), max_tokens=8192)
_rl._lf_base = _rl.llm_factory.func
EVAL.JUEZ_PAUSA_S = 6.0
print("Juez: max_tokens=8192 · pausa 6 s entre llamadas")


Juez: max_tokens=8192 · pausa 6 s entre llamadas


In [5]:
# ═══════════════════════════════════════════════════════════════
# E9-bootstrap · lo mínimo que E9 necesita, sin cargar el pipeline
# ═══════════════════════════════════════════════════════════════
# E9 no llama al recuperador ni a Ollama: solo lee las respuestas ya
# generadas desde Drive. Por eso no hace falta reconstruir el corpus.
# Córrela en un entorno SIN GPU, después de E1.
import json, shutil
import numpy as np, pandas as pd
from pathlib import Path

assert "EVAL" in globals(), "Corre primero E1 (configuración y montaje de Drive)."

def leer_jsonl(p):
    p = Path(p)
    datos = [json.loads(l) for l in open(p, encoding="utf-8")] if p.exists() else []
    return [f for f in datos if "deepseek" not in f.get("modelo", "")]   # exclusión por viabilidad

def media_boot(serie, b=2000, semilla=42):
    s = pd.Series(serie).dropna().astype(float).values
    if len(s) == 0:
        return "—"
    rng = np.random.default_rng(semilla)
    bs = rng.choice(s, (b, len(s))).mean(axis=1)
    return (f"{100*s.mean():.1f} [{100*np.percentile(bs, 2.5):.1f}–"
            f"{100*np.percentile(bs, 97.5):.1f}] (n={len(s)})").replace(".", ",")

GENERADOR = "qwen3:8b"          # el mismo de EVAL.GENERADOR_FAMILIA, fijado a mano

# --- banco: solo las columnas que E9 usa (sin corpus, sin norm_clave) ---
BANCO = pd.read_csv(EVAL.BANCO, dtype=str, keep_default_na=False)
BANCO["debe_abstenerse"] = BANCO.debe_abstenerse.str.lower().isin(["true", "1", "si", "sí"])

# --- respuestas generadas: se recuperan de Drive si no están en /content ---
LOCAL = Path(f"/content/respuestas_{EVAL.SPLIT}.jsonl")
DRIVE = EVAL.DIR_RES / LOCAL.name
if DRIVE.exists() and not LOCAL.exists():
    shutil.copy(DRIVE, LOCAL)
if not LOCAL.exists():
    raise SystemExit(f"⛔ No encuentro {DRIVE}. Sin las respuestas generadas no se "
                     f"puede puntuar: hay que correr E7 en un entorno con GPU.")

RESP = pd.DataFrame([f for f in leer_jsonl(LOCAL) if not f.get("error")])
RESP = RESP.merge(BANCO[["id", "debe_abstenerse"]], on="id", how="left")

huellas = set(RESP.get("huella", pd.Series(dtype=str)).dropna())
print(f"✅ RESP reconstruido: {len(RESP)} respuestas · generador {GENERADOR}")
print(f"   huella(s) de configuración: {huellas or '—'}")
if len(huellas) > 1:
    print("   ⚠️  Hay más de una configuración mezclada: revísalo antes de puntuar.")
print(RESP.groupby(["condicion", "modelo"]).size().to_frame("n").to_string())
print("\nSigue con E9-pre (configuración del juez) y después E9.")

✅ RESP reconstruido: 200 respuestas · generador qwen3:8b
   huella(s) de configuración: {'cd627464444c'}
                     n
condicion modelo      
C0        qwen3:8b  50
C1        qwen3:8b  50
C2        qwen3:8b  50
C2v10     qwen3:8b  50

Sigue con E9-pre (configuración del juez) y después E9.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⛔ E9 (VERSIÓN SUPERADA) — NO EJECUTAR
# ═══════════════════════════════════════════════════════════════
# Calcula CINCO métricas sobre TODAS las condiciones. La versión vigente
# es la celda siguiente, recortada a dos métricas sobre C1 y C2 para que
# quepa en la cuota gratuita. Correr esta encola cientos de evaluaciones
# y agota días de cuota.
# Se conserva como registro de qué se midió primero y por qué se recortó.
raise SystemExit("⛔ Celda superada. Usa la siguiente (E9 vigente). "
                 "Si de verdad quieres correrla, borra este raise.")

# ═══════════════════════════════════════════════════════════════
# E9 · RAGAS CON JUEZ EXTERNO (Gemini) — se reanuda sola
# ═══════════════════════════════════════════════════════════════
# El juez NO es ninguno de los generadores (evita el sesgo de autopreferencia).
# Métricas sin referencia: siempre. Con referencia (AnswerCorrectness,
# ContextRecall): solo en las filas donde el banco trae respuesta_referencia.
import asyncio
from google.colab import userdata
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.metrics.collections import (Faithfulness, AnswerRelevancy, ContextRelevance,
                                       AnswerCorrectness, ContextRecall)

cli_juez = AsyncOpenAI(base_url=EVAL.JUEZ_URL, api_key=userdata.get(EVAL.JUEZ_SECRETO))
juez = llm_factory(EVAL.JUEZ_MODELO, client=cli_juez)
emb_juez = OpenAIEmbeddings(client=cli_juez, model=EVAL.JUEZ_EMB)

# prueba de conexión: falla rápido si la clave o el nombre del modelo están mal
_ = await cli_juez.chat.completions.create(model=EVAL.JUEZ_MODELO, max_tokens=5,
                                           messages=[{"role": "user", "content": "ok"}])
print(f"⚖️  Juez: {EVAL.JUEZ_MODELO} · embeddings: {EVAL.JUEZ_EMB}")

# prueba de humo de RAGAS completo (salida estructurada + embeddings) antes de
# gastar cuota: si esto falla, el problema es de configuración, no de los datos.
_f = await Faithfulness(llm=juez).ascore(
    user_input="¿Qué es la Agencia?", response="La ley crea la Agencia de Protección de Datos Personales.",
    retrieved_contexts=["Créase la Agencia de Protección de Datos Personales."])
_r = await AnswerRelevancy(llm=juez, embeddings=emb_juez).ascore(
    user_input="¿Qué es la Agencia?", response="La ley crea la Agencia de Protección de Datos Personales.")
print(f"   prueba de humo · faithfulness={_f.value:.2f} · answer_relevancy={_r.value:.2f}")

METRICAS = {
    "faithfulness":       (Faithfulness(llm=juez), True,  False),   # (objeto, requiere_contexto, requiere_referencia)
    "answer_relevancy":   (AnswerRelevancy(llm=juez, embeddings=emb_juez), False, False),
    "context_relevance":  (ContextRelevance(llm=juez), True, False),
    "answer_correctness": (AnswerCorrectness(llm=juez, embeddings=emb_juez), False, True),
    "context_recall":     (ContextRecall(llm=juez), True, True),
}
REF = dict(zip(BANCO.id, BANCO.respuesta_referencia))

async def puntuar(nombre, m, f):
    kw = dict(user_input=f.pregunta, response=f.respuesta, retrieved_contexts=f.contextos,
              reference=REF.get(f.id, ""))
    firma = {"faithfulness": ("user_input", "response", "retrieved_contexts"),
             "answer_relevancy": ("user_input", "response"),
             "context_relevance": ("user_input", "retrieved_contexts"),
             "answer_correctness": ("user_input", "response", "reference"),
             "context_recall": ("user_input", "retrieved_contexts", "reference")}[nombre]
    return (await m.ascore(**{k: kw[k] for k in firma})).value

R_LOCAL = Path(f"/content/ragas_{EVAL.SPLIT}.jsonl"); R_DRIVE = EVAL.DIR_RES / R_LOCAL.name
if R_DRIVE.exists() and not R_LOCAL.exists():
    shutil.copy(R_DRIVE, R_LOCAL)
hechas_r = {(x["condicion"], x["modelo"], x["id"], x["metrica"])
            for x in leer_jsonl(R_LOCAL) if x.get("valor") is not None}

tareas = []
for f in RESP[~RESP.debe_abstenerse].itertuples():
    for nombre, (m, req_ctx, req_ref) in METRICAS.items():
        if (req_ctx and not f.contextos) or (req_ref and not REF.get(f.id, "").strip()):
            continue
        if (f.condicion, f.modelo, f.id, nombre) not in hechas_r:
            tareas.append((nombre, m, f))
print(f"Pendientes: {len(tareas)} evaluaciones")

for j, (nombre, m, f) in enumerate(tareas, 1):
    fila = dict(condicion=f.condicion, modelo=f.modelo, id=f.id, metrica=nombre, valor=None)
    for intento in range(3):
        try:
            fila["valor"] = float(await puntuar(nombre, m, f)); break
        except Exception as e:
            fila["error"] = f"{type(e).__name__}: {str(e)[:120]}"
            await asyncio.sleep(10 * (intento + 1))       # cuota de la API
    with open(R_LOCAL, "a", encoding="utf-8") as h:
        h.write(json.dumps(fila, ensure_ascii=False) + "\n")
    shutil.copy(R_LOCAL, R_DRIVE)
    if j % 20 == 0 or fila["valor"] is None:
        print(f"[{j}/{len(tareas)}] {f.condicion} {f.modelo} {f.id} {nombre}: {fila['valor']}")
    await asyncio.sleep(EVAL.JUEZ_PAUSA_S)

RAG = pd.DataFrame([x for x in leer_jsonl(R_LOCAL) if x.get("valor") is not None])
RAG = RAG.drop_duplicates(["condicion", "modelo", "id", "metrica"], keep="last")
T_RAGAS = RAG.pivot_table(index=["condicion", "modelo"], columns="metrica", values="valor",
                          aggfunc=media_boot)
print("═"*72 + "\nRAGAS (×100, media [IC 95 % bootstrap])\n" + "═"*72)
display(T_RAGAS)

In [7]:
# ═══════════════════════════════════════════════════════════════
# E9 · RAGAS CON JUEZ EXTERNO (Gemini) — se reanuda sola
# ═══════════════════════════════════════════════════════════════
# ESTA es la versión vigente (la celda anterior quedó superada).
# Dos métricas, solo C1 y C2 con el generador principal: el recorte
# es para que quepa en la cuota gratuita.
#
# Reanudable: cada evaluación se escribe al .jsonl y se copia a Drive.
# Al re-ejecutar, 'hechas_r' salta las ya hechas. NO cambiar
# EVAL.JUEZ_MODELO entre corridas: puntajes de jueces distintos no son
# comparables entre sí.
async def _e9b():
    global cli_juez, juez, emb_juez, METRICAS, REF, RAG, T_RAGAS, R_LOCAL, R_DRIVE
    # ═══════════════════════════════════════════════════════════════
    # E9 · RAGAS CON JUEZ EXTERNO (Gemini) — se reanuda sola
    # ═══════════════════════════════════════════════════════════════
    # El juez NO es ninguno de los generadores (evita el sesgo de autopreferencia).
    # Métricas sin referencia: siempre. Con referencia (AnswerCorrectness,
    # ContextRecall): solo en las filas donde el banco trae respuesta_referencia.
    import asyncio
    from google.colab import userdata
    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.embeddings import OpenAIEmbeddings
    from ragas.metrics.collections import (Faithfulness, AnswerRelevancy, ContextRelevance,
                                           AnswerCorrectness, ContextRecall)

    cli_juez = AsyncOpenAI(base_url=EVAL.JUEZ_URL, api_key=userdata.get(EVAL.JUEZ_SECRETO))
    juez = llm_factory(EVAL.JUEZ_MODELO, client=cli_juez, max_tokens=8192)
    emb_juez = OpenAIEmbeddings(client=cli_juez, model=EVAL.JUEZ_EMB)

    # prueba de conexión: falla rápido si la clave o el nombre del modelo están mal
    _ = await cli_juez.chat.completions.create(model=EVAL.JUEZ_MODELO, max_tokens=5,
                                               messages=[{"role": "user", "content": "ok"}])
    print(f"⚖️  Juez: {EVAL.JUEZ_MODELO} · embeddings: {EVAL.JUEZ_EMB}")

    # prueba de humo de RAGAS completo (salida estructurada + embeddings) antes de
    # gastar cuota: si esto falla, el problema es de configuración, no de los datos.
    _f = await Faithfulness(llm=juez).ascore(
        user_input="¿Qué es la Agencia?", response="La ley crea la Agencia de Protección de Datos Personales.",
        retrieved_contexts=["Créase la Agencia de Protección de Datos Personales."])
    _r = await AnswerRelevancy(llm=juez, embeddings=emb_juez).ascore(
        user_input="¿Qué es la Agencia?", response="La ley crea la Agencia de Protección de Datos Personales.")
    print(f"   prueba de humo · faithfulness={_f.value:.2f} · answer_relevancy={_r.value:.2f}")

    METRICAS = {
        "faithfulness":     (Faithfulness(llm=juez), True,  False),
        "answer_relevancy": (AnswerRelevancy(llm=juez, embeddings=emb_juez), False, False),
    }
    REF = dict(zip(BANCO.id, BANCO.respuesta_referencia))

    async def puntuar(nombre, m, f):
        kw = dict(user_input=f.pregunta, response=f.respuesta, retrieved_contexts=f.contextos,
                  reference=REF.get(f.id, ""))
        firma = {"faithfulness": ("user_input", "response", "retrieved_contexts"),
                 "answer_relevancy": ("user_input", "response"),
                 "context_relevance": ("user_input", "retrieved_contexts"),
                 "answer_correctness": ("user_input", "response", "reference"),
                 "context_recall": ("user_input", "retrieved_contexts", "reference")}[nombre]
        return (await m.ascore(**{k: kw[k] for k in firma})).value

    R_LOCAL = Path(f"/content/ragas_{EVAL.SPLIT}.jsonl"); R_DRIVE = EVAL.DIR_RES / R_LOCAL.name
    if R_DRIVE.exists() and not R_LOCAL.exists():
        shutil.copy(R_DRIVE, R_LOCAL)
    hechas_r = {(x["condicion"], x["modelo"], x["id"], x["metrica"])
                for x in leer_jsonl(R_LOCAL) if x.get("valor") is not None}

    tareas = []
    for f in RESP[(~RESP.debe_abstenerse) & RESP.condicion.isin(["C1", "C2"]) & (RESP.modelo == GENERADOR)].itertuples():
        for nombre, (m, req_ctx, req_ref) in METRICAS.items():
            if (req_ctx and not f.contextos) or (req_ref and not REF.get(f.id, "").strip()):
                continue
            if (f.condicion, f.modelo, f.id, nombre) not in hechas_r:
                tareas.append((nombre, m, f))
    # Intercalar por condición: si la cuota se agota a mitad, la pérdida se
    # reparte entre C1 y C2 en vez de concentrarse en la que va al final.
    # Sin esto, la corrida anterior dejó C1 con n=45 y C2 con n=28-32, y
    # ningún contraste C2 vs C1 era interpretable.
    from collections import defaultdict
    from itertools import zip_longest
    _grupos = defaultdict(list)
    for t in tareas:
        _grupos[t[2].condicion].append(t)
    tareas = [t for tanda in zip_longest(*[_grupos[k] for k in sorted(_grupos)])
                for t in tanda if t is not None]
    print(f"Pendientes: {len(tareas)} evaluaciones · "
          + " · ".join(f"{k}={len(v)}" for k, v in sorted(_grupos.items())))

    for j, (nombre, m, f) in enumerate(tareas, 1):
        fila = dict(condicion=f.condicion, modelo=f.modelo, id=f.id, metrica=nombre, valor=None)
        for intento in range(3):
            try:
                fila["valor"] = float(await puntuar(nombre, m, f)); break
            except Exception as e:
                fila["error"] = f"{type(e).__name__}: {str(e)[:120]}"
                await asyncio.sleep(10 * (intento + 1))       # cuota de la API
        with open(R_LOCAL, "a", encoding="utf-8") as h:
            h.write(json.dumps(fila, ensure_ascii=False) + "\n")
        shutil.copy(R_LOCAL, R_DRIVE)
        if j % 20 == 0 or fila["valor"] is None:
            print(f"[{j}/{len(tareas)}] {f.condicion} {f.modelo} {f.id} {nombre}: {fila['valor']}")
        await asyncio.sleep(EVAL.JUEZ_PAUSA_S)

    RAG = pd.DataFrame([x for x in leer_jsonl(R_LOCAL) if x.get("valor") is not None])
    RAG = RAG[RAG.metrica.isin(list(METRICAS))]
    RAG = RAG.drop_duplicates(["condicion", "modelo", "id", "metrica"], keep="last")
    T_RAGAS = RAG.pivot_table(index=["condicion", "modelo"], columns="metrica", values="valor",
                              aggfunc=media_boot)
    print("═"*72 + "\nRAGAS (×100, media [IC 95 % bootstrap])\n" + "═"*72)
    display(T_RAGAS)
    T_RAGAS.to_csv(EVAL.DIR_RES / f"tabla_ragas_{EVAL.SPLIT}.csv")

await _e9b()


⚖️  Juez: gemini-3.5-flash-lite · embeddings: gemini-embedding-001
   prueba de humo · faithfulness=1.00 · answer_relevancy=0.63
Pendientes: 119 evaluaciones · C1=59 · C2=60


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 30.2300602s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit exc

[20/119] C2 qwen3:8b G203 answer_relevancy: 0.8223738356900965


ERROR:instructor.v2.retry:API call failed on attempt 3: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 25.486627883s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit e

[40/119] C2 qwen3:8b G268 answer_relevancy: 0.968984470463545


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 29.796301041s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit e

[60/119] C2 qwen3:8b G328 answer_relevancy: 0.8401128465026889


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 28.927540514s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit e

[80/119] C2 qwen3:8b G379 answer_relevancy: 0.9002156114538424
[100/119] C2 qwen3:8b G445 answer_relevancy: 0.8624879026726195
════════════════════════════════════════════════════════════════════════
RAGAS (×100, media [IC 95 % bootstrap])
════════════════════════════════════════════════════════════════════════


,metrica,answer_relevancy,faithfulness
condicion,modelo,,
C1,qwen3:8b,"87,4 [85,3–89,4] (n=45)","71,1 [63,7–78,1] (n=45)"
C2,qwen3:8b,"87,0 [85,1–88,8] (n=45)","67,4 [61,6–73,0] (n=45)"


In [16]:
# ═══════════════════════════════════════════════════════════════
# E10 · CONTRASTE DE HIPÓTESIS + TABLAS PARA LA TESIS
# ═══════════════════════════════════════════════════════════════
# Pruebas PAREADAS (mismas preguntas en ambas condiciones):
#  · binarias (cita inventada, cita esperada) → McNemar exacta. Para datos
#    0/1 pareados es la prueba estándar; equivale a la prueba de signos, que
#    es en lo que se reduce Wilcoxon cuando las diferencias son ±1.
#  · continuas (RAGAS) → Wilcoxon de rangos con signo.
#  · Corrección de Holm sobre toda la familia de pruebas.
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

def pares(df, col, a, b, modelo):
    x = df[(df.condicion == a) & (df.modelo == modelo)].set_index("id")[col]
    y = df[(df.condicion == b) & (df.modelo == modelo)].set_index("id")[col]
    j = pd.concat([x, y], axis=1, keys=["a", "b"]).dropna()
    return j.a, j.b

pruebas = []
def probar(h, metrica, a, b, df, binaria):
    xa, xb = pares(df, metrica, a, b, GENERADOR)
    if len(xa) < 5:
        return
    if binaria:
        xa, xb = xa.astype(bool), xb.astype(bool)
        tabla = [[(xa & xb).sum(), (xa & ~xb).sum()], [(~xa & xb).sum(), (~xa & ~xb).sum()]]
        p = mcnemar(tabla, exact=True).pvalue
        dif = f"{100*(xa.mean()-xb.mean()):+.1f} pp"
    else:
        d = xa - xb
        p = wilcoxon(xa, xb).pvalue if (d != 0).any() else 1.0
        dif = f"{100*d.median():+.1f} (mediana)"
    pruebas.append(dict(hipotesis=h, metrica=metrica, comparacion=f"{a} vs {b}", n=len(xa),
                        diferencia=dif, p=p, prueba="McNemar exacta" if binaria else "Wilcoxon"))

for h, b in [("H1", "C0"), ("H2", "C1")]:
    for m in ["cita_inventada", "cita_esperada"]:
        probar(h, m, "C2", b, MET, binaria=True)
    if "RAG" in dir() and len(RAG):
        R = RAG.pivot_table(index=["condicion", "modelo", "id"], columns="metrica",
                            values="valor").reset_index()
        for m in ["answer_relevancy", "faithfulness", "context_relevance", "answer_correctness"]:
            if m in R and not (b == "C0" and m in ("faithfulness", "context_relevance")):
                probar(h, m, "C2", b, R, binaria=False)

# H2 también a nivel de recuperación (Recall@k final, C2 vs C1)
rc = DF_RECALL.pivot_table(index="id", columns="etapa", values="hit")
if len(rc):
    a, b = rc["C2 · final (top-k)"].astype(bool), rc["C1 · densa fija (top-k)"].astype(bool)
    tabla = [[(a & b).sum(), (a & ~b).sum()], [(~a & b).sum(), (~a & ~b).sum()]]
    pruebas.append(dict(hipotesis="H2", metrica="recall@k", comparacion="C2 vs C1", n=len(a),
                        diferencia=f"{100*(a.mean()-b.mean()):+.1f} pp",
                        p=mcnemar(tabla, exact=True).pvalue, prueba="McNemar exacta"))

T_HIP = pd.DataFrame(pruebas)
if len(T_HIP):
    T_HIP["p_holm"] = multipletests(T_HIP.p, method="holm")[1]
    T_HIP["significativa (α=0,05)"] = T_HIP.p_holm < 0.05
    print("═"*72 + "\nCONTRASTE DE HIPÓTESIS (pruebas pareadas, corrección de Holm)\n" + "═"*72)
    display(T_HIP.round(4))

# H3 · brecha de lenguaje: misma pregunta en versión coloquial (Q) y jurídica (J)
pares_j = BANCO[BANCO.tipo == "juridica"][["id", "par_id"]]
fin = DF_RECALL[DF_RECALL.etapa == "C2 · final (top-k)"].set_index("id").hit
sinx = DF_RECALL[DF_RECALL.etapa == "C2 sin expansión · final"].set_index("id").hit
h3 = pares_j.assign(juridica_final=pares_j.id.map(fin), coloquial_final=pares_j.par_id.map(fin),
                    coloquial_sin_exp=pares_j.par_id.map(sinx)).dropna()
if len(h3):
    print("\nH3 · Recall@k en pares coloquial/jurídica (C2):")
    display(pd.DataFrame({c: [pct(h3[c])] for c in
                          ["coloquial_sin_exp", "coloquial_final", "juridica_final"]}, index=["recall"]))

# ---------- exportar ----------
for nombre, t in [("tabla_recall", T_RECALL), ("tabla_metricas_auto", T_AUTO),
                  ("tabla_hipotesis", T_HIP if len(T_HIP) else None),
                  ("tabla_ragas", T_RAGAS if "T_RAGAS" in dir() else None)]:
    if t is None:
        continue
    t.to_csv(EVAL.DIR_RES / f"{nombre}_{EVAL.SPLIT}.csv")
    try:
        (EVAL.DIR_RES / f"{nombre}_{EVAL.SPLIT}.tex").write_text(t.to_latex(), encoding="utf-8")
    except Exception:
        pass

meta = dict(split=EVAL.SPLIT, huella=HUELLA, commit=COMMIT, parche_p1=EVAL.APLICAR_PARCHE_P1,
            n_chunks_c2=N_CHUNKS, n_fragmentos_c1=len(CHUNKS_C1), generador=GENERADOR,
            modelos=list(MODELOS.values()), juez=EVAL.JUEZ_MODELO,
            n_preguntas=len(EVAL_SET), fecha=datetime.datetime.now().isoformat(timespec="seconds"))
(EVAL.DIR_RES / f"metadatos_{EVAL.SPLIT}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
print(f"\n💾 Tablas (.csv + .tex) y metadatos en {EVAL.DIR_RES}")

════════════════════════════════════════════════════════════════════════
CONTRASTE DE HIPÓTESIS (pruebas pareadas, corrección de Holm)
════════════════════════════════════════════════════════════════════════


,hipotesis,metrica,comparacion,n,diferencia,p,prueba,p_holm,"significativa (α=0,05)"
0,H1,cita_inventada,C2 vs C0,50,-22.0 pp,0.0010,McNemar exacta,0.0039,True
1,H1,cita_esperada,C2 vs C0,45,+84.4 pp,0.0000,McNemar exacta,0.0000,True
2,H2,cita_inventada,C2 vs C1,50,-2.0 pp,1.0000,McNemar exacta,1.0000,False
3,H2,cita_esperada,C2 vs C1,45,+6.7 pp,0.3750,McNemar exacta,0.7500,False
4,H2,recall@k,C2 vs C1,45,+8.9 pp,0.2188,McNemar exacta,0.6562,False



💾 Tablas (.csv + .tex) y metadatos en /content/drive/MyDrive/Material tesis - Ley Protección Datos/resultados evaluacion tesis/v11


In [ ]:
# ═══════════════════════════════════════════════════════════════
# E11 · CALIBRACIÓN DEL UMBRAL DEL REORDENADOR
# ═══════════════════════════════════════════════════════════════
# Solo sobre el split de DESARROLLO. El de prueba no se toca acá.
# CALIBRACIÓN DEL UMBRAL DEL REORDENADOR · solo split de desarrollo
# Criterio fijado ANTES de ver resultados: máximo Recall@k en preguntas respondibles;
# empate → umbral más alto (más conservador). Se reporta además el efecto en fuera de corpus.
import time
_orig = CFG.UMBRAL_RERANK
UMBRALES = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.10]
filas = []; t0 = time.time()
for u in UMBRALES:
    CFG.UMBRAL_RERANK = u
    for r in EVAL_SET.itertuples():
        fin, _ = construir_contexto(r.pregunta, verbose=False)
        lista = [c for c, _ in fin]
        hit, rr = acierto(lista, r.gold) if r.gold else (np.nan, np.nan)
        filas.append(dict(umbral=u, id=r.id, abst=bool(r.debe_abstenerse), gold=bool(r.gold),
                          n=len(lista), hit=hit, rr=rr, smax=max([s for _, s in fin], default=0.0)))
CFG.UMBRAL_RERANK = _orig
CAL = pd.DataFrame(filas); CAL.to_csv(EVAL.DIR_RES / "calibracion_umbral_desarrollo.csv", index=False)
resp = CAL[CAL.gold & ~CAL.abst]; fuera = CAL[CAL.abst]
T_CAL = pd.DataFrame({
    "recall@k": resp.groupby("umbral").hit.mean().mul(100).round(1),
    "MRR": resp.groupby("umbral").rr.mean().round(3),
    "respondibles_sin_fuentes": resp.groupby("umbral").n.apply(lambda x: int((x == 0).sum())),
    "fuera_corpus_sin_fuentes": fuera.groupby("umbral").n.apply(lambda x: int((x == 0).sum())),
    "n_resp": resp.groupby("umbral").size(), "n_fuera": fuera.groupby("umbral").size()})
mejor = T_CAL["recall@k"].max(); ELEGIDO = max(T_CAL.index[T_CAL["recall@k"] == mejor])
print(f"⏱️ {time.time()-t0:.0f}s"); print(T_CAL.to_string())
print(f"\nCriterio → umbral elegido: {ELEGIDO} (actual {_orig})")
sin = CAL[(CAL.umbral == 0.0)].set_index("id").loc[["Q27","Q33","Q37","Q43"], ["smax","n","hit"]]
print("\nPreguntas que hoy quedan sin fuentes (con umbral 0):\n", sin.to_string())
T_CAL.to_csv(EVAL.DIR_RES / "tabla_calibracion_umbral.csv")


⏱️ 831s
         recall@k    MRR  respondibles_sin_fuentes  fuera_corpus_sin_fuentes  n_resp  n_fuera
umbral                                                                                       
0.000   68.888889  0.476                         0                         0      45        5
0.005   68.888889  0.476                         0                         3      45        5
0.010   66.666667  0.471                         0                         3      45        5
0.020   64.444444  0.460                         0                         3      45        5
0.030   62.222222  0.449                         2                         3      45        5
0.050   62.222222  0.448                         4                         3      45        5
0.100   55.555556  0.400                         7                         4      45        5

Criterio → umbral elegido: 0.005 (actual 0.05)

Preguntas que hoy quedan sin fuentes (con umbral 0):
          smax  n    hit
id                 

## Notas para el equipo

- **Etiquetas del banco.** `articulos_esperados` son una **propuesta** (columna `fuente_etiqueta`), con `confianza_etiqueta` alta/media/baja. Antes de la corrida de `prueba`, revisen al menos las de confianza media y baja. Si luego hay abogados, que validen el banco y llenen `respuesta_referencia`: E9 activa sola AnswerCorrectness y ContextRecall.
- **Sin abogados**, la "exactitud" del póster se reporta con *cita del artículo esperado* (E8) y Recall@k (E5). Kappa entre abogados queda como trabajo futuro o limitación.
- **Detectores por reglas** (abstención, parser de citas): reporten su precisión con la muestra de `auditoria_manual_*.csv`.
- **Citas a leyes fuera del corpus** (p. ej. una "Ley 20.222" en C0) se cuentan como *no verificables*, no como inventadas: la tasa de citas inventadas de C0 es una **cota inferior**. Revísenlas en `refs_externas` de `metricas_auto_*.csv`.
- **Mantengan el computador despierto** durante E7 (en Mac: Ajustes → Batería/Pantalla → evitar reposo, o `caffeinate -d` en Terminal). Si se desconecta, vuelvan a ejecutar todo: E7 retoma donde quedó.
- **Reproducibilidad:** cada fila guarda `huella` (configuración) y `commit` (código). No mezclar huellas en un mismo análisis.